# AnatomyLocked Character Studio

## Overview

This notebook is a **persistent human character studio** designed for:
- Reference-grade anatomy
- Identity-locked characters
- Regional anatomical refinement
- Pose-accurate deformation
- Reloadable, reusable characters
- Multi-character scene composition

This is **not** a random image generator.  
Once a character is finalized, they must only change in ways a real human could.

## Quick Identity Guide

**Must never change**
- Bone structure, facial geometry, body proportions
- Permanent skin features (freckles, moles, scars, birthmarks) and exact placement

**Allowed to vary**
- Pose/posture, muscle flexion/compression, skin folds due to movement
- Hair style (same root pattern unless explicitly changed)
- Makeup, nail color, lighting, camera angle, optional clothing

**Realistic deformation is expected**
- Stretching, bending, and flexing should change appearance only in ways a real body would

## Identity Lock Helper (Quick Settings)

- Fix the base seed for the character and keep it constant for identity-locked renders.
- Use the same identity embedding and reference set across all variants.
- Lock completed regions while tuning others (region-by-region refinement).
- Change only one factor at a time (pose, lighting, or camera) to isolate drift.

## Required Output Set (Reference Table)

| Set | Purpose | Views / Notes |
| --- | ------- | ------------- |
| Neutral | Baseline anatomy | Front, back, left, right |
| 3/4 | Shape consistency | Front 3/4 and back 3/4 |
| Poses | Deformation realism | Standing, sitting, crouched, dynamic |
| Lighting | Form clarity | Key, fill, rim; soft and hard |
| Scenes (opt) | Storytelling | Optional environment renders |

## Drift Triage Checklist

- Confirm base model, ControlNet stack, and seed match the baseline.
- Compare invariant features (moles, scars, facial proportions) against the baseline set.
- If drift appears, change only one variable at a time (pose, lighting, or camera).
- Re-run a neutral view to verify the lock before generating variants.
- If drift persists, refresh the identity embedding reference set.

In [4]:
from pathlib import Path
import copy
import json

template_path = Path("Character_Validation_History_Template.json")

if template_path.exists():
    print(f"Template already exists: {template_path.resolve()}")
else:
    template = {
        "character_id": "CH-0001",
        "created_date": "YYYY-MM-DD",
        "baseline": {
            "base_model": "",
            "controlnet_stack": {
                "pose": "",
                "depth": "",
                "normal": ""
            },
            "seed": 0,
            "identity_embedding": {
                "type": "",
                "reference_images": []
            }
        },
        "identity_lock": {
            "locked_regions": ["face", "torso"],
            "notes": ""
        },
        "render_sets": [
            {
                "set_id": "SET-0001",
                "date": "YYYY-MM-DD",
                "purpose": "neutral views",
                "lighting": "",
                "camera": "",
                "pose_pack": "",
                "outputs": {
                    "image_paths": []
                },
                "validation": {
                    "identity_invariants_pass": False,
                    "anatomy_accuracy_pass": False,
                    "notes": ""
                }
            }
        ],
        "issues": [
            {
                "date": "YYYY-MM-DD",
                "issue": "",
                "resolution": ""
            }
        ],
        "next_steps": ""
    }
    template_path.write_text(json.dumps(template, indent=2), encoding="utf-8")
    print(f"Created template: {template_path.resolve()}")

Template already exists: /content/Character_Validation_History_Template.json


In [5]:
from datetime import date
from pathlib import Path
import json

template_path = Path("Character_Validation_History_Template.json")
output_dir = Path("character_histories")
output_dir.mkdir(exist_ok=True)

character_id = "CH-0001"
created_date = date.today().isoformat()

if not template_path.exists():
    print(f"Template not found: {template_path.resolve()}")
    print("Skipping history file creation. Add the template and re-run this cell.")
else:
    with template_path.open("r", encoding="utf-8") as handle:
        data = json.load(handle)

    data["character_id"] = character_id
    data["created_date"] = created_date

    output_path = output_dir / f"{character_id}_history.json"
    with output_path.open("w", encoding="utf-8") as handle:
        json.dump(data, handle, indent=2)

    print(f"Wrote {output_path}")

Wrote character_histories/CH-0001_history.json


---

## Core Rules

### Identity Invariants (Must Never Change)
- Bone structure
- Facial geometry
- Body proportions
- Permanent skin features (freckles, moles, scars, birthmarks)
- Relative placement and shape of invariant features

### Allowed Variations
- Pose and posture
- Muscle flexion and compression
- Skin folding due to movement
- Hair style (root pattern remains consistent)
- Makeup and nail color
- Lighting and camera angle
- Clothing (optional)

### Forbidden Variations
- Face drift
- Proportion changes
- Feature relocation
- Anatomy exaggeration
- Stylization that breaks realism




---

## Expected Output

Each character produces a reusable reference package:
- Neutral anatomy views
- Pose variations
- Lighting variants
- Scene renders
- Reloadable identity data

---

# SECTION 1 — Environment Setup & Dependencies

**Purpose:**  
Prepare Colab environment, GPU, and required libraries.

- Python version check
- GPU availability
- Dependency installation
- Cache and output directories

📌 1.1 Mount Google Drive

In [6]:
from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


📌 1.2 Define AI Workspace Paths

In [7]:
from pathlib import Path

# Base AI workspace
AI_BASE = Path("/content/drive/My Drive/AI")

# High-level directories
AI_DIRS = {
    "datasets": AI_BASE / "datasets",
    "experiments": AI_BASE / "Experiments",
    "images": AI_BASE / "Images",
    "rag": AI_BASE / "rag",
    "training": AI_BASE / "Training",
    "models": AI_BASE / "models",
}

# Model subdirectories (UNDER models/)
MODEL_DIRS = {
    # Diffusion base models live under models/diffusion_base/{sd, sdxl}
    "diffusion_base": AI_DIRS["models"] / "diffusion_base",
    "diffusion_sd": AI_DIRS["models"] / "diffusion_base" / "sd",
    "diffusion_sdxl": AI_DIRS["models"] / "diffusion_base" / "sdxl",
    # Back-compat alias for older cells
    "base_models": AI_DIRS["models"] / "diffusion_base",
    "audio_models": AI_DIRS["models"] / "audio_models",
    "checkpoints": AI_DIRS["models"] / "checkpoints",
    "controlnet": AI_DIRS["models"] / "controlnet",
    "llm": AI_DIRS["models"] / "llm",
    "loras": AI_DIRS["models"] / "loras",
}


In [8]:
# Guard: ensure Drive + base folders exist
if not AI_BASE.exists():
    raise FileNotFoundError(
        f"AI_BASE not found: {AI_BASE}. Did you mount Google Drive?"
    )

print("AI_BASE:", AI_BASE)
print("Model directories:")
for key in (
    "diffusion_base",
    "diffusion_sd",
    "diffusion_sdxl",
    "controlnet",
    "loras",
    "checkpoints",
):
    print(f" - {key}: {MODEL_DIRS[key]}")

# Fail fast if base model folder is empty
if not MODEL_DIRS["diffusion_base"].exists():
    raise FileNotFoundError(
        f"Base model folder missing: {MODEL_DIRS['diffusion_base']}"
    )
if not any(MODEL_DIRS["diffusion_base"].rglob("*")):
    raise FileNotFoundError(
        "No model files found under diffusion_base. "
        "Add SD/SDXL model files and re-run."
    )


AI_BASE: /content/drive/My Drive/AI
Model directories:
 - diffusion_base: /content/drive/My Drive/AI/models/diffusion_base
 - diffusion_sd: /content/drive/My Drive/AI/models/diffusion_base/sd
 - diffusion_sdxl: /content/drive/My Drive/AI/models/diffusion_base/sdxl
 - controlnet: /content/drive/My Drive/AI/models/controlnet
 - loras: /content/drive/My Drive/AI/models/loras
 - checkpoints: /content/drive/My Drive/AI/models/checkpoints


<details>
<summary><strong>Optional: Model Downloads (Collapsed)</strong></summary>

Use this section only when you need to add new models. Supports Hugging Face, Civitai, Google Drive, and direct URLs.

In [9]:
_DOWNLOAD_DEPS_READY = False
_HF_HUB_AVAILABLE = False

def ensure_download_deps() -> bool:
    global _DOWNLOAD_DEPS_READY, _HF_HUB_AVAILABLE, requests, gdown, tqdm, hf_hub_download, HfHubHTTPError
    if _DOWNLOAD_DEPS_READY:
        return True
    try:
        import requests
        import gdown
        from tqdm.auto import tqdm
        from huggingface_hub import hf_hub_download
        from huggingface_hub.utils import HfHubHTTPError
        _DOWNLOAD_DEPS_READY = True
        _HF_HUB_AVAILABLE = True
        print("Download dependencies are available.")
        return True
    except ImportError:
        print("Installing download dependencies...")
        !pip install -q huggingface_hub gdown requests tqdm
        import requests
        import gdown
        from tqdm.auto import tqdm
        from huggingface_hub import hf_hub_download
        from huggingface_hub.utils import HfHubHTTPError
        _DOWNLOAD_DEPS_READY = True
        _HF_HUB_AVAILABLE = True
        print("Download dependencies installed.")
        return True
    except Exception as exc:
        _DOWNLOAD_DEPS_READY = False
        _HF_HUB_AVAILABLE = False
        print(f"Download dependency setup failed: {exc}")
        return False

try:
    from google.colab import userdata
except Exception:
    userdata = None

def _get_colab_secret(name: str) -> str:
    if userdata is None:
        print("Colab userdata not available.")
        return ""
    try:
        token = userdata.get(name)
        if token:
            print(f"✓ {name} retrieved from Colab secrets.")
            return token
        print(f"✗ {name} not found in Colab secrets.")
        return ""
    except Exception as exc:
        print(f"✗ Error retrieving {name}: {exc}")
        return ""

def get_hf_token() -> str:
    return _get_colab_secret("HF_TOKEN")

def get_civit_token() -> str:
    return _get_colab_secret("CIVIT_TOKEN")

from pathlib import Path
from urllib.parse import urlparse

def _is_hf_url(parsed_url) -> bool:
    host = parsed_url.netloc.lower()
    return "huggingface.co" in host or "hf.co" in host

def _classify_download_name(filename: str) -> str:
    name = filename.lower()
    if "sdxl" in name or "sd_xl" in name or "sd xl" in name:
        return "sdxl"
    if (
        "v1-5" in name
        or "v15" in name
        or "sd15" in name
        or "stable-diffusion-v1" in name
        or "sd-v1" in name
    ):
        return "sd"
    if "controlnet" in name or "openpose" in name or "depth" in name or "normal" in name:
        return "controlnet"
    if "lora" in name or "lyco" in name:
        return "lora"
    if "audio" in name or "whisper" in name or "tts" in name:
        return "audio"
    if name.endswith(".bin") or name.endswith(".gguf"):
        return "llm"
    if name.endswith(".ckpt") or name.endswith(".safetensors") or name.endswith(".pt") or name.endswith(".pth"):
        return "checkpoint"
    return "checkpoint"

In [10]:
def download_file_direct(url: str, destination_path: Path) -> bool:
    if not _DOWNLOAD_DEPS_READY:
        print("Download dependencies not ready. Run ensure_download_deps() first.")
        return False
    print("Attempting direct download.")
    headers = {}
    if "civitai.com" in url:
        civit_token = get_civit_token()
        if civit_token:
            headers["Authorization"] = f"Bearer {civit_token}"
            print("Added Civitai token to headers for authenticated download.")
        else:
            print("Civitai token not found, attempting unauthenticated download.")

    try:
        response = requests.get(url, stream=True, headers=headers)
        response.raise_for_status()
        total_size = int(response.headers.get("content-length", 0))

        with open(destination_path, "wb") as handle:
            with tqdm(total=total_size, unit="B", unit_scale=True, desc=destination_path.name) as pbar:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        handle.write(chunk)
                        pbar.update(len(chunk))
        return True
    except requests.exceptions.RequestException as exc:
        print(f"✗ Error during direct download: {exc}")
        if destination_path.exists():
            destination_path.unlink()
        return False

def download_file(url: str, destination_path: Path) -> bool:
    if not _DOWNLOAD_DEPS_READY:
        print("Download dependencies not ready. Run ensure_download_deps() first.")
        return False
    print(f"Attempting to download from {url} to {destination_path}")
    destination_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        parsed_url = urlparse(url)

        if _HF_HUB_AVAILABLE and _is_hf_url(parsed_url):
            print("Detected Hugging Face link.")
            hf_token = get_hf_token()
            path_parts = parsed_url.path.split("/")
            if len(path_parts) >= 5 and path_parts[3] == "resolve":
                repo_id = f"{path_parts[1]}/{path_parts[2]}"
                filename = path_parts[-1]
                if filename:
                    try:
                        downloaded_path = hf_hub_download(repo_id=repo_id, filename=filename, local_dir=destination_path.parent, local_dir_use_symlinks=False, token=hf_token if hf_token else None)
                        if Path(downloaded_path).name != destination_path.name:
                            Path(downloaded_path).rename(destination_path)
                        print(f"✓ Successfully downloaded {destination_path.name} from Hugging Face.")
                        return True
                    except HfHubHTTPError as exc:
                        print(f"✗ Error during Hugging Face download (HTTP Error): {exc}")
                        if "401 Client Error" in str(exc) or "403 Client Error" in str(exc):
                            print("    This might be a private model or require authentication. Check your HF_TOKEN.")
                        if destination_path.exists():
                            destination_path.unlink()
                        return False
                    except Exception as exc:
                        print(f"✗ Hugging Face download failed: {exc}. Attempting direct download.")
                        return download_file_direct(url, destination_path)
            print("Could not parse Hugging Face URL. Attempting direct download.")
            return download_file_direct(url, destination_path)

        if "drive.google.com" in url:
            print("Detected Google Drive link.")
            gdown.download(url, str(destination_path), quiet=False, fuzzy=True)
            print(f"✓ Successfully downloaded {destination_path.name}")
            return True

        return download_file_direct(url, destination_path)
    except gdown.exceptions.GDriveDownloadError as exc:
        print(f"✗ Error during Google Drive download: {exc}")
        if destination_path.exists():
            destination_path.unlink()
        return False
    except Exception as exc:
        print(f"✗ An unexpected error occurred: {exc}")
        if destination_path.exists():
            destination_path.unlink()
        return False

In [11]:
#@title Download model (optional)
ENABLE_MODEL_DOWNLOADS = False  #@param {type:"boolean"}
MODEL_URL = ""  #@param {type:"string"}
MODEL_TYPE_HINT = "auto"  #@param ["auto", "sd", "sdxl", "controlnet", "lora", "checkpoint", "llm", "audio"]

if ENABLE_MODEL_DOWNLOADS:
    if not MODEL_URL:
        print("Paste a model URL into MODEL_URL.")
    else:
        if ensure_download_deps():
            hint = None if MODEL_TYPE_HINT == "auto" else MODEL_TYPE_HINT
            smart_download_model(MODEL_URL, model_type_hint=hint)
        else:
            print("Download dependencies not ready.")
else:
    print("Downloads are disabled.")

Downloads are disabled.


</details>

📌 1.3 Create Missing Directories (Non-Destructive)

In [12]:
for name, path in AI_DIRS.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"✓ {name}: {path}")

✓ datasets: /content/drive/My Drive/AI/datasets
✓ experiments: /content/drive/My Drive/AI/Experiments
✓ images: /content/drive/My Drive/AI/Images
✓ rag: /content/drive/My Drive/AI/rag
✓ training: /content/drive/My Drive/AI/Training
✓ models: /content/drive/My Drive/AI/models


📌 1.4 Quick Sanity Check (Optional)

In [13]:
assert AI_BASE.exists(), "AI base directory was not created correctly."
print("AI workspace is ready.")

AI workspace is ready.


🔍 1.5 Scan Existing Models & Assets

In [14]:
def scan_directory(path, extensions=None):
    results = []
    if not path.exists():
        return results

    for p in path.rglob("*"):
        if p.is_file():
            if extensions is None or p.suffix.lower() in extensions:
                results.append(p)
    return results


📦 Scan Diffusion / ML Models

In [15]:
MODEL_EXTS = {".ckpt", ".safetensors", ".pt", ".pth"}

base_models = scan_directory(MODEL_DIRS["diffusion_base"], MODEL_EXTS)
loras = scan_directory(MODEL_DIRS["loras"], MODEL_EXTS)
controlnets = scan_directory(MODEL_DIRS["controlnet"], MODEL_EXTS)
checkpoints = scan_directory(MODEL_DIRS["checkpoints"], MODEL_EXTS)


print(f"Base models: {len(base_models)}")
print(f"LoRAs: {len(loras)}")
print(f"ControlNets: {len(controlnets)}")
print(f"Checkpoints: {len(checkpoints)}")


Base models: 3
LoRAs: 0
ControlNets: 7
Checkpoints: 1


📄 Optional: Print a Preview

In [16]:
def preview(files, limit=10):
    for f in files[:limit]:
        print(f" - {f.name}")

print("\nSample base models:")
preview(base_models)

print("\nSample LoRAs:")
preview(loras)



Sample base models:
 - v1-5-pruned.safetensors
 - v1-5-pruned-emaonly.safetensors
 - sd_xl_base_1.0.safetensors

Sample LoRAs:


---

# SECTION 2 — Base Model & Control Stack Selection

**Purpose:**  
Define the foundational models used throughout the notebook.

Includes:
- Base diffusion model (photorealistic, anatomy-capable)
- ControlNet modules (pose, depth, normals)
- Identity embedding models
- Version locking and rationale

This section should rarely change.

2.1 Model Registry Data Structures

In [17]:
from dataclasses import dataclass
from pathlib import Path
from typing import List, Dict


In [18]:
@dataclass
class ModelEntry:
    name: str
    path: Path
    model_type: str     # e.g. "sd", "sdxl", "controlnet", "lora"
    size_mb: float


In [19]:
MODEL_REGISTRY: Dict[str, List[ModelEntry]] = {
    "sd": [],
    "sdxl": [],
    "controlnet": [],
    "lora": [],
    "checkpoint": [],
    "llm": [],
    "audio": [],
}


🔍 2.2 Helper: File Size Utility

In [20]:
def file_size_mb(path: Path) -> float:
    return round(path.stat().st_size / (1024 ** 2), 2)


🔍 2.3 Heuristic Model Classifier

In [57]:
def classify_model(path: Path) -> str:
    name = path.name.lower()
    parent_hint = "/".join(part.lower() for part in path.parts)

    if "/diffusion_base/sd/" in parent_hint:
        return "sd"
    if "/diffusion_base/sdxl/" in parent_hint:
        return "sdxl"
    if "/controlnet/" in parent_hint:
        return "controlnet"
    if "/loras/" in parent_hint:
        return "lora"
    if "/llm/" in parent_hint:
        return "llm"
    if "/audio_models/" in parent_hint:
        return "audio"

    if "controlnet" in name or "openpose" in name or "depth" in name or "normal" in name:
        return "controlnet"
    if "lora" in name or "lyco" in name:
        return "lora"
    if "audio" in name or "whisper" in name or "tts" in name:
        return "audio"
    if "sdxl" in name or "sd_xl" in name or "sd xl" in name:
        return "sdxl"
    if (
        "v1-5" in name
        or "v15" in name
        or "sd15" in name
        or "sd_1.5" in name
        or "sd-1.5" in name
        or "stable-diffusion-v1" in name
        or "sd-v1" in name
    ):
        return "sd"

    if path.suffix in {".ckpt", ".safetensors", ".pt", ".pth"}:
        return "checkpoint"
    if path.suffix in {".bin", ".gguf"}:
        return "llm"

    return "unknown"


🔍 2.4 Scan Model Directories & Populate Registry

In [22]:
MODEL_EXTS = {".ckpt", ".safetensors", ".pt", ".pth", ".bin", ".gguf"}


In [58]:
MODEL_EXTS = {".ckpt", ".safetensors", ".pt", ".pth", ".bin", ".gguf"}

# Clear registry to avoid stale counts
for _k in list(MODEL_REGISTRY.keys()):
    MODEL_REGISTRY[_k] = []

def register_models_from_dir(
    directory: Path,
    model_type_override: str | None = None,
    skip_subdirs: tuple[str, ...] = (),
):
    for p in directory.rglob("*"):
        if p.is_file() and p.suffix.lower() in MODEL_EXTS:
            parent_hint = "/".join(part.lower() for part in p.parts)
            if skip_subdirs and any(s in parent_hint for s in skip_subdirs):
                continue
            mtype = model_type_override or classify_model(p)
            if mtype in MODEL_REGISTRY:
                MODEL_REGISTRY[mtype].append(
                    ModelEntry(
                        name=p.name,
                        path=p,
                        model_type=mtype,
                        size_mb=file_size_mb(p)
                    )
                )

# Scan model subdirectories only
for key in [
    "diffusion_base",
    "diffusion_sd",
    "diffusion_sdxl",
    "checkpoints",
    "controlnet",
    "loras",
    "llm",
    "audio_models",
]:
    override = None
    skip_subdirs = ()
    if key == "diffusion_sd":
        override = "sd"
    elif key == "diffusion_sdxl":
        override = "sdxl"
    elif key == "diffusion_base":
        # Avoid double-counting if files are already in sd/sdxl subfolders
        skip_subdirs = ("/diffusion_base/sd/", "/diffusion_base/sdxl/")
    register_models_from_dir(MODEL_DIRS[key], override, skip_subdirs)

In [59]:
def preflight_checks(base_model_entry=None, use_controlnet=False, controlnet_entries=None):
    errors = []
    try:
        import torch
        if not torch.cuda.is_available():
            errors.append("CUDA not available. Ensure a GPU runtime is selected.")
    except Exception as exc:
        errors.append(f"Torch not available: {exc}")

    has_base = bool(MODEL_REGISTRY.get("sd")) or bool(MODEL_REGISTRY.get("sdxl"))
    if not has_base:
        errors.append(
            "No SD/SDXL base models found in the registry. "
            "Add models under models/diffusion_base/sd or models/diffusion_base/sdxl."
        )

    if base_model_entry is not None:
        try:
            if not Path(base_model_entry.path).exists():
                errors.append(f"Selected base model missing: {base_model_entry.path}")
        except Exception as exc:
            errors.append(f"Could not verify selected base model: {exc}")

    if use_controlnet:
        available = []
        if controlnet_entries:
            available = [e for e in controlnet_entries.values() if e is not None]
        if not available:
            errors.append(
                "USE_CONTROLNET is True but no ControlNet files are available. "
                "Add ControlNet weights or set USE_CONTROLNET=False."
            )

    if errors:
        print("Preflight checks failed:\n")
        for err in errors:
            print(f" - {err}")
        raise RuntimeError("Preflight checks failed. See messages above.")

    print("Preflight checks passed.")
    return True


In [60]:
# Scan model subdirectories only
for key in [
    "diffusion_sd",
    "diffusion_sdxl",
    "checkpoints",
    "controlnet",
    "loras",
    "llm",
    "audio_models",
]:
    override = None
    if key == "diffusion_sd":
        override = "sd"
    elif key == "diffusion_sdxl":
        override = "sdxl"
    register_models_from_dir(MODEL_DIRS[key], override)

In [61]:
# List files in models/diffusion_base
base_dir = MODEL_DIRS["diffusion_base"]
print(f"diffusion_base: {base_dir}")

if not base_dir.exists():
    print("  (folder not found)")
else:
    files = [p for p in base_dir.rglob("*") if p.is_file()]
    if not files:
        print("  (no files found)")
    else:
        for p in sorted(files):
            rel = p.relative_to(base_dir)
            print(f"  - {rel}")


diffusion_base: /content/drive/My Drive/AI/models/diffusion_base
  - sd/v1-5-pruned-emaonly.safetensors
  - sd/v1-5-pruned.safetensors
  - sdxl/sd_xl_base_1.0.safetensors


📊 2.5 Registry Summary

In [62]:
def print_registry_summary():
    print("📦 Model Registry Summary\n")
    for k, v in MODEL_REGISTRY.items():
        print(f"{k.upper():12s}: {len(v)} models")

print_registry_summary()


📦 Model Registry Summary

SD          : 4 models
SDXL        : 2 models
CONTROLNET  : 14 models
LORA        : 0 models
CHECKPOINT  : 2 models
LLM         : 38 models
AUDIO       : 12 models


In [63]:
# Diagnostic: Base models in diffusion_base and their classification
_diag_files = scan_directory(MODEL_DIRS["diffusion_base"], MODEL_EXTS)
print("\nDiagnostic base models in diffusion_base:")
if not _diag_files:
    print("  (none found)")
else:
    for _p in _diag_files:
        print(f"  {_p.name} -> {classify_model(_p)}")



Diagnostic base models in diffusion_base:
  v1-5-pruned.safetensors -> sd
  v1-5-pruned-emaonly.safetensors -> sd
  sd_xl_base_1.0.safetensors -> sdxl


## Model Availability Snapshot (Run to Refresh)

Run the next cell to see what models and ControlNets are already available in this workspace.

In [64]:
from IPython.display import Markdown, display

def build_registry_snapshot(limit: int = 5) -> str:
    rows = []
    for model_type, entries in MODEL_REGISTRY.items():
        if entries:
            names = ", ".join([e.name for e in entries[:limit]])
            if len(entries) > limit:
                names = f"{names} (+{len(entries) - limit} more)"
        else:
            names = "None"
        rows.append((model_type, len(entries), names))

    lines = ["| Type | Count | Sample |", "| --- | --- | --- |"]
    for model_type, count, names in rows:
        lines.append(f"| {model_type} | {count} | {names} |")
    return "\n".join(lines)

display(Markdown(build_registry_snapshot()))

| Type | Count | Sample |
| --- | --- | --- |
| sd | 4 | v1-5-pruned.safetensors, v1-5-pruned-emaonly.safetensors, v1-5-pruned.safetensors, v1-5-pruned-emaonly.safetensors |
| sdxl | 2 | sd_xl_base_1.0.safetensors, sd_xl_base_1.0.safetensors |
| controlnet | 14 | OpenPoseXL2.safetensors, controlnet-openpose-sdxl-1.0.safetensors, controlnet-depth-sdxl-1.0.safetensors, controlnet-union-sdxl-1.0.safetensors, control_v11p_sd15_openpose.safetensors (+9 more) |
| lora | 0 | None |
| checkpoint | 2 | Illustrious-XL-v1.0.safetensors, Illustrious-XL-v1.0.safetensors |
| llm | 38 | model-00007-of-00019.safetensors, model-00008-of-00019.safetensors, model-00002-of-00019.safetensors, model-00006-of-00019.safetensors, model-00005-of-00019.safetensors (+33 more) |
| audio | 12 | model.safetensors, model.bin, model.bin, model.bin, pytorch_model.bin (+7 more) |

In [65]:
from pathlib import Path

snapshot_path = Path("model_registry_snapshot.md")
snapshot_path.write_text(build_registry_snapshot(), encoding="utf-8")
print(f"Wrote {snapshot_path}")

Wrote model_registry_snapshot.md


📄 2.6 Preview Models (Human-Readable)

In [66]:
def preview_registry(model_type: str, limit: int = 10):
    entries = MODEL_REGISTRY.get(model_type, [])
    if not entries:
        print(f"No models registered for type: {model_type}")
        return

    print(f"\n{model_type.upper()} MODELS:")
    for e in entries[:limit]:
        print(f" - {e.name} ({e.size_mb} MB)")


In [67]:
preview_registry("sd")
preview_registry("sdxl")
preview_registry("controlnet")
preview_registry("lora")


SD MODELS:
 - v1-5-pruned.safetensors (7346.46 MB)
 - v1-5-pruned-emaonly.safetensors (4067.56 MB)
 - v1-5-pruned.safetensors (7346.46 MB)
 - v1-5-pruned-emaonly.safetensors (4067.56 MB)

SDXL MODELS:
 - sd_xl_base_1.0.safetensors (6616.67 MB)
 - sd_xl_base_1.0.safetensors (6616.67 MB)

CONTROLNET MODELS:
 - OpenPoseXL2.safetensors (4772.35 MB)
 - controlnet-openpose-sdxl-1.0.safetensors (2386.23 MB)
 - controlnet-depth-sdxl-1.0.safetensors (4772.35 MB)
 - controlnet-union-sdxl-1.0.safetensors (2395.66 MB)
 - control_v11p_sd15_openpose.safetensors (1378.21 MB)
 - control_v11f1p_sd15_depth.safetensors (1378.21 MB)
 - control_v11p_sd15_normalbae.safetensors (1378.21 MB)
 - OpenPoseXL2.safetensors (4772.35 MB)
 - controlnet-openpose-sdxl-1.0.safetensors (2386.23 MB)
 - controlnet-depth-sdxl-1.0.safetensors (4772.35 MB)
No models registered for type: lora


### Recommended Base + ControlNet Sources (SDXL + SD1.5)

Place files in the model folders shown below. Use the optional download section to fetch any URL.

- SDXL base -> models/diffusion_base/sdxl
  https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors
- SDXL ControlNet OpenPose -> models/controlnet
  https://huggingface.co/diffusers/controlnet-openpose-sdxl-1.0/resolve/main/diffusion_pytorch_model.safetensors
- SDXL ControlNet Depth -> models/controlnet
  https://huggingface.co/diffusers/controlnet-depth-sdxl-1.0/resolve/main/diffusion_pytorch_model.safetensors
- SDXL ControlNet Normal -> models/controlnet
  https://huggingface.co/diffusers/controlnet-normal-sdxl-1.0/resolve/main/diffusion_pytorch_model.safetensors

- SD1.5 base -> models/diffusion_base/sd
  https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors
- SD1.5 ControlNet OpenPose -> models/controlnet
  https://huggingface.co/lllyasviel/control_v11p_sd15_openpose/resolve/main/diffusion_pytorch_model.safetensors
- SD1.5 ControlNet Depth -> models/controlnet
  https://huggingface.co/lllyasviel/control_v11f1p_sd15_depth/resolve/main/diffusion_pytorch_model.safetensors
- SD1.5 ControlNet Normal -> models/controlnet
  https://huggingface.co/lllyasviel/control_v11p_sd15_normalbae/resolve/main/diffusion_pytorch_model.safetensors


⭐ 2.7 Default Model Selection

In [68]:
DEFAULT_MODELS = {
    "base_sd": None,
    "base_sdxl": None,
    "sd_controlnet_pose": None,
    "sd_controlnet_depth": None,
    "sd_controlnet_normal": None,
    "sdxl_controlnet_pose": None,
    "sdxl_controlnet_depth": None,
    "sdxl_controlnet_normal": None,
}


In [69]:
def _detect_family_from_entry(entry: ModelEntry) -> str:
    if entry.model_type in {"sd", "sdxl"}:
        return entry.model_type
    lowered = entry.name.lower()
    if "sdxl" in lowered or "sd_xl" in lowered or "sd xl" in lowered:
        return "sdxl"
    if "sd15" in lowered or "sd_1.5" in lowered or "sd-1.5" in lowered:
        return "sd"
    return "unknown"

def select_default(model_type: str, contains: str, family: str | None = None):
    for m in MODEL_REGISTRY.get(model_type, []):
        if contains.lower() in m.name.lower():
            if family:
                m_family = _detect_family_from_entry(m)
                if m_family != family:
                    continue
            return m
    return None


Example defaults (adjust names as needed)

In [70]:
DEFAULT_MODELS["base_sdxl"] = select_default("sdxl", "base")
DEFAULT_MODELS["base_sd"] = select_default("sd", "v1-5") or select_default("sd", "sd15")

DEFAULT_MODELS["sdxl_controlnet_pose"] = select_default("controlnet", "openpose", "sdxl")
DEFAULT_MODELS["sdxl_controlnet_depth"] = select_default("controlnet", "depth", "sdxl")
DEFAULT_MODELS["sdxl_controlnet_normal"] = (
    select_default("controlnet", "normal", "sdxl")
    or select_default("controlnet", "union", "sdxl")
 )

DEFAULT_MODELS["sd_controlnet_pose"] = select_default("controlnet", "openpose", "sd")
DEFAULT_MODELS["sd_controlnet_depth"] = select_default("controlnet", "depth", "sd")
DEFAULT_MODELS["sd_controlnet_normal"] = select_default("controlnet", "normal", "sd")

ACTIVE_BASE_FAMILY = "sdxl" if DEFAULT_MODELS["base_sdxl"] else "sd" if DEFAULT_MODELS["base_sd"] else None

if ACTIVE_BASE_FAMILY == "sdxl":
    ACTIVE_CONTROLNET_DEFAULTS = {
        "pose": DEFAULT_MODELS["sdxl_controlnet_pose"],
        "depth": DEFAULT_MODELS["sdxl_controlnet_depth"],
        "normal": DEFAULT_MODELS["sdxl_controlnet_normal"],
    }
elif ACTIVE_BASE_FAMILY == "sd":
    ACTIVE_CONTROLNET_DEFAULTS = {
        "pose": DEFAULT_MODELS["sd_controlnet_pose"],
        "depth": DEFAULT_MODELS["sd_controlnet_depth"],
        "normal": DEFAULT_MODELS["sd_controlnet_normal"],
    }
else:
    ACTIVE_CONTROLNET_DEFAULTS = {"pose": None, "depth": None, "normal": None}

In [71]:
print("⭐ Default Model Selection\n")
for k, v in DEFAULT_MODELS.items():
    print(f"{k:22s}: {v.name if v else 'None'}")

print("\nActive base family:", ACTIVE_BASE_FAMILY or "None")

⭐ Default Model Selection

base_sd               : v1-5-pruned.safetensors
base_sdxl             : sd_xl_base_1.0.safetensors
sd_controlnet_pose    : control_v11p_sd15_openpose.safetensors
sd_controlnet_depth   : control_v11f1p_sd15_depth.safetensors
sd_controlnet_normal  : control_v11p_sd15_normalbae.safetensors
sdxl_controlnet_pose  : controlnet-openpose-sdxl-1.0.safetensors
sdxl_controlnet_depth : controlnet-depth-sdxl-1.0.safetensors
sdxl_controlnet_normal: controlnet-union-sdxl-1.0.safetensors

Active base family: sdxl


In [72]:
def _detect_family(value) -> str:
    if isinstance(value, ModelEntry):
        if value.model_type in {"sd", "sdxl"}:
            return value.model_type
        lowered = value.name.lower()
    else:
        lowered = str(value).lower()
    if "sdxl" in lowered or "sd_xl" in lowered or "sd xl" in lowered:
        return "sdxl"
    if "sd15" in lowered or "sd_1.5" in lowered or "sd-1.5" in lowered:
        return "sd"
    return "unknown"

def warn_controlnet_compatibility(base_entry, control_entries):
    if base_entry is None:
        print("Base model not set. Skipping compatibility checks.")
        return

    base_family = _detect_family(base_entry)
    if base_family == "unknown":
        print(f"Base model family unknown: {base_entry.name}")
        return

    for key, entry in control_entries.items():
        if entry is None:
            continue
        control_family = _detect_family(entry)
        if control_family == "unknown":
            print(f"ControlNet {key} family unknown: {entry.name}")
            continue
        if control_family != base_family:
            print(
                f"⚠️ ControlNet {key} looks like {control_family} but base is {base_family}. "
                "Expect poor results or load errors."
            )

print("\nCompatibility checks:")
warn_controlnet_compatibility(
    DEFAULT_MODELS.get("base_sdxl"),
    {
        "pose": DEFAULT_MODELS.get("sdxl_controlnet_pose"),
        "depth": DEFAULT_MODELS.get("sdxl_controlnet_depth"),
        "normal": DEFAULT_MODELS.get("sdxl_controlnet_normal"),
    }
)
warn_controlnet_compatibility(
    DEFAULT_MODELS.get("base_sd"),
    {
        "pose": DEFAULT_MODELS.get("sd_controlnet_pose"),
        "depth": DEFAULT_MODELS.get("sd_controlnet_depth"),
        "normal": DEFAULT_MODELS.get("sd_controlnet_normal"),
    }
)


Compatibility checks:


In [73]:
# Runtime configuration
# Set USE_CONTROLNET = True only after a successful base render.
USE_CONTROLNET = True

# Default output size for SDXL without ControlNet
OUTPUT_WIDTH = 768
OUTPUT_HEIGHT = 1024

# Optional: map controlnet keys to HF repo IDs for auto-download
# Example:
# CONTROLNET_REPO_IDS = {
#     "pose": "diffusers/controlnet-openpose-sdxl-1.0",
#     "depth": "diffusers/controlnet-depth-sdxl-1.0",
#     "normal": "diffusers/controlnet-normal-sdxl-1.0",
# }
CONTROLNET_REPO_IDS = {}

# Optional: per-controlnet conditioning scales
CONTROLNET_CONDITIONING_SCALES = {
    "pose": 1.0,
    "depth": 0.8,
    "normal": 0.8,
}


# **🔹 SECTION 2.8 — Diffusers Pipeline Initialization (Registry-Driven)**

**2.8.1 Install Diffusers Stack (Once)**

To ensure a `pip install` command runs only if necessary, you can check for the presence of a key module before executing the installation. This makes your notebook more efficient and prevents unnecessary re-installations. Here's an example using the `diffusers` library:

In [90]:
try:
    import diffusers
    import transformers
    from transformers import CLIPImageProcessor
    print("diffusers/transformers are available.")
except Exception as exc:
    print(f"Updating diffusers stack (missing CLIPImageProcessor): {exc}")
    !pip install -q -U diffusers transformers accelerate xformers safetensors torchvision
    print("Installation complete.")

diffusers/transformers are available.


**2.8.2 Imports & Accelerator Setup**

In [91]:
import torch
import os
from diffusers import (
    StableDiffusionPipeline,
    StableDiffusionControlNetPipeline,
    StableDiffusionXLPipeline,
    StableDiffusionXLControlNetPipeline,
    ControlNetModel
)
from diffusers.utils import load_image


In [92]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print(f"Using device: {DEVICE}, dtype: {DTYPE}")


Using device: cuda, dtype: torch.float16


**2.8.3 Resolve Base Model from Registry**

In [93]:
def require_model(entry, label):
    if entry is None:
        raise RuntimeError(f"Required model missing: {label}")
    return str(entry.path)

In [95]:
def infer_base_family(model_entry):
    if model_entry is None:
        return None
    if model_entry.model_type in {"sd", "sdxl"}:
        return model_entry.model_type
    lowered = model_entry.name.lower()
    if "sdxl" in lowered or "sd_xl" in lowered or "sd xl" in lowered:
        return "sdxl"
    return "sd"


def build_base_model_options():
    options = []
    for model_type in ("sd", "sdxl"):
        for entry in MODEL_REGISTRY.get(model_type, []):
            label = f"{model_type.upper()} • {entry.name} ({entry.size_mb} MB)"
            options.append((label, entry))
    return options


BASE_MODEL_OPTIONS = build_base_model_options()
if not BASE_MODEL_OPTIONS:
    raise RuntimeError("No base SD/SDXL models were found in the model registry.")

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except Exception:
    pass

try:
    import ipywidgets as widgets
    from IPython.display import display

    base_model_widget = widgets.Dropdown(
        options=BASE_MODEL_OPTIONS,
        value=(DEFAULT_MODELS["base_sdxl"] or DEFAULT_MODELS["base_sd"] or BASE_MODEL_OPTIONS[0][1]),
        description="Base Model:",
        layout=widgets.Layout(width="95%"),
        style={"description_width": "initial"},
    )

    print("Available base models:")
    for idx, (label, _entry) in enumerate(BASE_MODEL_OPTIONS):
        print(f"  [{idx}] {label}")

    display(base_model_widget)
    print("If no dropdown appears in your environment, set BASE_MODEL_INDEX manually.")

    BASE_MODEL_INDEX = None  # Set to an integer index to force a model in non-widget environments.
    if isinstance(BASE_MODEL_INDEX, int) and 0 <= BASE_MODEL_INDEX < len(BASE_MODEL_OPTIONS):
        SELECTED_BASE_MODEL = BASE_MODEL_OPTIONS[BASE_MODEL_INDEX][1]
        print(f"Using manual base model index: {BASE_MODEL_INDEX}")
    else:
        SELECTED_BASE_MODEL = base_model_widget.value
except Exception as exc:
    print(f"⚠️ ipywidgets unavailable; falling back to automatic model selection. ({exc})")
    print("Available base models:")
    for idx, (label, _entry) in enumerate(BASE_MODEL_OPTIONS):
        print(f"  [{idx}] {label}")
    SELECTED_BASE_MODEL = DEFAULT_MODELS["base_sdxl"] or DEFAULT_MODELS["base_sd"] or BASE_MODEL_OPTIONS[0][1]

BASE_MODEL_PATH = require_model(SELECTED_BASE_MODEL, "Selected base model")
BASE_MODEL_FAMILY = infer_base_family(SELECTED_BASE_MODEL)

if BASE_MODEL_FAMILY == "sdxl":
    SELECTED_CONTROLNET_MODELS = {
        "pose": DEFAULT_MODELS["sdxl_controlnet_pose"],
        "depth": DEFAULT_MODELS["sdxl_controlnet_depth"],
        "normal": DEFAULT_MODELS["sdxl_controlnet_normal"],
    }
else:
    SELECTED_CONTROLNET_MODELS = {
        "pose": DEFAULT_MODELS["sd_controlnet_pose"],
        "depth": DEFAULT_MODELS["sd_controlnet_depth"],
        "normal": DEFAULT_MODELS["sd_controlnet_normal"],
    }

print(f"Selected base model: {SELECTED_BASE_MODEL.name}")
print(f"Detected base family: {BASE_MODEL_FAMILY}")
for c_key, c_entry in SELECTED_CONTROLNET_MODELS.items():
    print(f"ControlNet {c_key:6s}: {c_entry.name if c_entry else 'None'}")

preflight_checks(SELECTED_BASE_MODEL, USE_CONTROLNET, SELECTED_CONTROLNET_MODELS)


Available base models:
  [0] SD • v1-5-pruned.safetensors (7346.46 MB)
  [1] SD • v1-5-pruned-emaonly.safetensors (4067.56 MB)
  [2] SD • v1-5-pruned.safetensors (7346.46 MB)
  [3] SD • v1-5-pruned-emaonly.safetensors (4067.56 MB)
  [4] SDXL • sd_xl_base_1.0.safetensors (6616.67 MB)
  [5] SDXL • sd_xl_base_1.0.safetensors (6616.67 MB)


Dropdown(description='Base Model:', index=4, layout=Layout(width='95%'), options=(('SD • v1-5-pruned.safetenso…

If no dropdown appears in your environment, set BASE_MODEL_INDEX manually.
Selected base model: sd_xl_base_1.0.safetensors
Detected base family: sdxl
ControlNet pose  : controlnet-openpose-sdxl-1.0.safetensors
ControlNet depth : controlnet-depth-sdxl-1.0.safetensors
ControlNet normal: controlnet-union-sdxl-1.0.safetensors
Preflight checks passed.


True

**2.8.4 Load ControlNet Models (If Available)**

In [96]:
CONTROLNETS = {}


def load_controlnet(key, model_entry):
    repo_id = CONTROLNET_REPO_IDS.get(key) if isinstance(CONTROLNET_REPO_IDS, dict) else None

    if model_entry is None and not repo_id:
        print(f"ControlNet {key} not found and no repo id provided.")
        return None

    if model_entry is not None:
        print(f"Loading ControlNet from file: {model_entry.name}")
        try:
            return ControlNetModel.from_single_file(
                model_entry.path,
                torch_dtype=DTYPE,
            )
        except Exception as exc:
            print(
                f"ControlNet {key} single-file load failed: {exc}. "
                "Some SDXL ControlNet weights require a config.json."
            )
            if not repo_id:
                print(
                    "Provide a ControlNet repo id in CONTROLNET_REPO_IDS "
                    "or download the full diffusers folder."
                )
                return None

    if repo_id:
        print(f"Loading ControlNet from repo: {repo_id}")
        try:
            return ControlNetModel.from_pretrained(
                repo_id,
                torch_dtype=DTYPE,
            )
        except Exception as exc:
            print(f"ControlNet {key} repo load failed: {exc}")
            return None

    return None


if USE_CONTROLNET:
    CONTROLNETS["pose"] = load_controlnet("pose", SELECTED_CONTROLNET_MODELS.get("pose"))
    CONTROLNETS["depth"] = load_controlnet("depth", SELECTED_CONTROLNET_MODELS.get("depth"))
    CONTROLNETS["normal"] = load_controlnet("normal", SELECTED_CONTROLNET_MODELS.get("normal"))
    ACTIVE_CONTROLNETS = [cn for cn in CONTROLNETS.values() if cn is not None]
    if not ACTIVE_CONTROLNETS:
        print("No ControlNets loaded. Proceeding without ControlNet.")
else:
    print("USE_CONTROLNET is False. Skipping ControlNet loading.")
    ACTIVE_CONTROLNETS = []

print(f"Active ControlNets: {len(ACTIVE_CONTROLNETS)}")


Loading ControlNet from file: controlnet-openpose-sdxl-1.0.safetensors
Loading ControlNet from file: controlnet-depth-sdxl-1.0.safetensors
Loading ControlNet from file: controlnet-union-sdxl-1.0.safetensors


Some weights of the model checkpoint were not used when initializing ControlNetModel: 
 ['control_add_embedding.linear_1.bias, control_add_embedding.linear_1.weight, control_add_embedding.linear_2.bias, control_add_embedding.linear_2.weight, spatial_ch_projs.bias, spatial_ch_projs.weight, task_embedding, transformer_layes.0.attn.in_proj_bias, transformer_layes.0.attn.in_proj_weight, transformer_layes.0.attn.out_proj.bias, transformer_layes.0.attn.out_proj.weight, transformer_layes.0.ln_1.bias, transformer_layes.0.ln_1.weight, transformer_layes.0.ln_2.bias, transformer_layes.0.ln_2.weight, transformer_layes.0.mlp.c_fc.bias, transformer_layes.0.mlp.c_fc.weight, transformer_layes.0.mlp.c_proj.bias, transformer_layes.0.mlp.c_proj.weight']


Active ControlNets: 3


In [97]:
# ControlNet loading is handled above.


In [98]:
ACTIVE_CONTROLNETS = [cn for cn in CONTROLNETS.values() if cn is not None]
print(f"Active ControlNets: {len(ACTIVE_CONTROLNETS)}")


Active ControlNets: 3


**2.8.5 Initialize the Pipeline**

In [99]:
is_single_checkpoint = os.path.isfile(BASE_MODEL_PATH)

if BASE_MODEL_FAMILY == "sdxl":
    if ACTIVE_CONTROLNETS:
        if is_single_checkpoint:
            pipe = StableDiffusionXLControlNetPipeline.from_single_file(
                BASE_MODEL_PATH,
                controlnet=ACTIVE_CONTROLNETS,
                torch_dtype=DTYPE,
                use_safetensors=True,
            )
        else:
            pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
                BASE_MODEL_PATH,
                controlnet=ACTIVE_CONTROLNETS,
                torch_dtype=DTYPE,
                safety_checker=None,
                variant="fp16"
            )
    else:
        if is_single_checkpoint:
            pipe = StableDiffusionXLPipeline.from_single_file(
                BASE_MODEL_PATH,
                torch_dtype=DTYPE,
                use_safetensors=True,
            )
        else:
            pipe = StableDiffusionXLPipeline.from_pretrained(
                BASE_MODEL_PATH,
                torch_dtype=DTYPE,
                safety_checker=None,
                variant="fp16"
            )
else:
    if ACTIVE_CONTROLNETS:
        pipe = StableDiffusionControlNetPipeline.from_single_file(
            BASE_MODEL_PATH,
            controlnet=ACTIVE_CONTROLNETS,
            torch_dtype=DTYPE,
            use_safetensors=True,
        )
    else:
        pipe = StableDiffusionPipeline.from_single_file(
            BASE_MODEL_PATH,
            torch_dtype=DTYPE,
            use_safetensors=True,
        )


Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

In [100]:
try:
    import xformers
    print("xformers is already installed.")
except ImportError:
    print("xformers not found, installing...")
    !pip install -q xformers==0.0.32.post2
    print("xformers installation complete.")

xformers is already installed.


In [101]:
pipe.to(DEVICE)

try:
    pipe.enable_xformers_memory_efficient_attention()
    print("✅ xFormers attention enabled.")
except ModuleNotFoundError:
    print("⚠️ xformers is not installed; running without memory-efficient attention.")
except Exception as exc:
    print(f"⚠️ Could not enable xformers attention: {exc}")

try:
    pipe.enable_model_cpu_offload()
except Exception as exc:
    print(f"⚠️ CPU offload could not be enabled: {exc}")


def print_runtime_package_versions():
    import importlib

    packages = [
        "torch",
        "torchvision",
        "xformers",
        "transformers",
        "diffusers",
        "accelerate",
        "bitsandbytes",
    ]

    for package_name in packages:
        try:
            module = importlib.import_module(package_name)
            version = getattr(module, "__version__", "unknown")
            print(f"{package_name:14s}: {version}")
        except ModuleNotFoundError:
            print(f"{package_name:14s}: not installed")


print("\nRuntime package versions:")
print_runtime_package_versions()


⚠️ xformers is not installed; running without memory-efficient attention.

Runtime package versions:
torch         : 2.9.0+cu128
torchvision   : 0.24.0+cu128
xformers      : 0.0.32.post2
transformers  : 5.0.0
diffusers     : 0.36.0
accelerate    : 1.12.0
bitsandbytes  : not installed


In [102]:
import torch, torchvision
print(torch.__version__)
print(torchvision.__version__)

2.9.0+cu128
0.24.0+cu128


**2.8.6 Deterministic Seeding (Critical for Identity Work)**

In [103]:
import random
import numpy as np

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


**2.8.7 Smoke Test (Minimal Render)**

In [104]:
set_seed(12345)

prompt = (
    "neutral anatomical reference photograph of a nude female in her mid 20s, "
    "standing relaxed, evenly lit, realistic proportions"
)

image = pipe(
    prompt=prompt,
    num_inference_steps=30,
    guidance_scale=5.5,
    width=OUTPUT_WIDTH,
    height=OUTPUT_HEIGHT,
).images[0]

display(image)


TypeError: For multiple controlnets: `image` must be type `list`

---

# SECTION 3 — Exploration Mode (Free Variation Playground)

**Purpose:**  
Generate broad anatomical variations to discover promising candidates.

Characteristics:
- Controlled randomness
- No identity locking
- Multiple seeds
- Neutral reference lighting

**Output:**  
A set of candidate images for review.

### Prompt Additions (Optional)

Add extra descriptors separated by commas (e.g., "athletic build, short hair").

### Quick First Render (No ControlNet)

Run this cell to validate the base pipeline before any ControlNet use.


In [ ]:
# Quick First Render (No ControlNet)
# This cell intentionally avoids ControlNet. If ControlNet is enabled,
# set USE_CONTROLNET=False and re-run Section 2.8 before running this cell.
if ACTIVE_CONTROLNETS:
    print("ControlNet is currently active. Set USE_CONTROLNET=False and re-run Section 2.8 to disable ControlNet.")
else:
    set_seed(20240217)
    quick_prompt = (
        "neutral anatomical reference photograph color of an adult nude asian teenage female front view, "
        "standing relaxed, evenly lit, realistic proportions"
    )
    quick_negative = "stylized, cartoon, deformed, extra limbs, low quality, blurry, lowres"
    quick_image = pipe(
        prompt=quick_prompt,
        negative_prompt=quick_negative,
        num_inference_steps=25,
        guidance_scale=5.5,
        width=OUTPUT_WIDTH,
        height=OUTPUT_HEIGHT,
    ).images[0]
    display(quick_image)


ControlNet is currently active. Set USE_CONTROLNET=False and re-run Section 2.8 to disable ControlNet.


: 

In [ ]:
try:
    import ipywidgets as widgets
except ImportError:
    print("ipywidgets not found, installing...")
    !pip install -q ipywidgets
    import ipywidgets as widgets

from IPython.display import display

prompt_additions_text = widgets.Textarea(
    value="",
    placeholder="Comma-separated additions for the prompt...",
    description="Prompt +:",
    layout=widgets.Layout(width="80%", height="80px")
)

negative_additions_text = widgets.Textarea(
    value="",
    placeholder="Comma-separated additions for the negative prompt...",
    description="Negative +:",
    layout=widgets.Layout(width="80%", height="80px")
)

run_button = widgets.Button(description="Run", button_style="primary")
run_output = widgets.Output()

PROMPT_ADDITIONS = []
NEGATIVE_PROMPT_ADDITIONS = []

def _parse_additions(text: str):
    return [item.strip() for item in text.split(",") if item.strip()]

def _run_clicked(_):
    global PROMPT_ADDITIONS, NEGATIVE_PROMPT_ADDITIONS
    PROMPT_ADDITIONS = _parse_additions(prompt_additions_text.value)
    NEGATIVE_PROMPT_ADDITIONS = _parse_additions(negative_additions_text.value)
    with run_output:
        run_output.clear_output()
        print("Prompt additions set for next run.")
        print("Negative additions set for next run.")
    prompt_additions_text.value = ""
    negative_additions_text.value = ""

run_button.on_click(_run_clicked)
display(widgets.VBox([prompt_additions_text, negative_additions_text, run_button, run_output]))

: 

In [ ]:
from datetime import datetime
from pathlib import Path
import subprocess
import sys
import json

import numpy as np
from IPython.display import display
from PIL import Image, ImageDraw


def ensure_controlnet_aux_installed():
    try:
        import controlnet_aux  # noqa: F401
        print("controlnet_aux is already installed.")
    except ImportError:
        print("Installing controlnet_aux preprocessors...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "controlnet_aux", "opencv-python"])
        print("controlnet_aux installation complete.")


def make_sample_pose_image(width: int = 768, height: int = 1024) -> Image.Image:
    canvas = Image.new("RGB", (width, height), (245, 245, 245))
    draw = ImageDraw.Draw(canvas)

    cx = width // 2
    head_r = width // 15
    shoulder_y = int(height * 0.24)
    hip_y = int(height * 0.50)
    knee_y = int(height * 0.72)
    foot_y = int(height * 0.92)

    draw.ellipse((cx - head_r, shoulder_y - 2 * head_r, cx + head_r, shoulder_y), outline=(20, 20, 20), width=12)
    draw.line((cx, shoulder_y, cx, hip_y), fill=(20, 20, 20), width=14)

    arm_span = width // 5
    elbow_drop = height // 10
    hand_drop = height // 7
    draw.line((cx, shoulder_y, cx - arm_span, shoulder_y + elbow_drop), fill=(20, 20, 20), width=12)
    draw.line((cx - arm_span, shoulder_y + elbow_drop, cx - int(arm_span * 0.6), shoulder_y + hand_drop), fill=(20, 20, 20), width=10)
    draw.line((cx, shoulder_y, cx + arm_span, shoulder_y + elbow_drop), fill=(20, 20, 20), width=12)
    draw.line((cx + arm_span, shoulder_y + elbow_drop, cx + int(arm_span * 0.6), shoulder_y + hand_drop), fill=(20, 20, 20), width=10)

    leg_span = width // 10
    draw.line((cx, hip_y, cx - leg_span, knee_y), fill=(20, 20, 20), width=13)
    draw.line((cx - leg_span, knee_y, cx - int(leg_span * 1.3), foot_y), fill=(20, 20, 20), width=11)
    draw.line((cx, hip_y, cx + leg_span, knee_y), fill=(20, 20, 20), width=13)
    draw.line((cx + leg_span, knee_y, cx + int(leg_span * 1.3), foot_y), fill=(20, 20, 20), width=11)

    return canvas


def generate_controlnet_conditioning_maps(source_image: Image.Image):
    from controlnet_aux import MidasDetector, NormalBaeDetector, OpenposeDetector

    openpose_detector = OpenposeDetector.from_pretrained("lllyasviel/ControlNet")
    depth_detector = MidasDetector.from_pretrained("lllyasviel/Annotators")
    normal_detector = NormalBaeDetector.from_pretrained("lllyasviel/Annotators")

    pose_map = openpose_detector(source_image)
    depth_map = depth_detector(source_image)
    normal_map = normal_detector(source_image)

    return {
        "pose": pose_map.convert("RGB"),
        "depth": depth_map.convert("RGB"),
        "normal": normal_map.convert("RGB"),
    }


EXPLORATION_DIR = AI_DIRS["images"] / "exploration" / datetime.utcnow().strftime("%Y%m%d_%H%M%S")
EXPLORATION_DIR.mkdir(parents=True, exist_ok=True)

conditioning_by_key = {}
active_controlnet_keys = []
source_image = None
source_path = None

if USE_CONTROLNET:
    ensure_controlnet_aux_installed()
    source_image = make_sample_pose_image(width=OUTPUT_WIDTH, height=OUTPUT_HEIGHT)
    source_path = EXPLORATION_DIR / "controlnet_source_pose.png"
    source_image.save(source_path)
    print(f"Saved source pose image: {source_path}")

    conditioning_by_key = generate_controlnet_conditioning_maps(source_image)
    for key, cond_image in conditioning_by_key.items():
        cond_path = EXPLORATION_DIR / f"conditioning_{key}.png"
        cond_image.save(cond_path)
        print(f"Saved conditioning map ({key}): {cond_path}")

    active_controlnet_keys = [key for key, model in CONTROLNETS.items() if model is not None]
    if active_controlnet_keys:
        print(f"Using ControlNet conditioning maps for: {', '.join(active_controlnet_keys)}")
    else:
        print("No active ControlNets found; exploration will run without ControlNet conditioning.")
else:
    print("USE_CONTROLNET is False. Exploration will run without ControlNet.")

base_prompt = (
    "neutral anatomical reference photograph of an adult human, "
    "standing relaxed, evenly lit, realistic proportions"
)
base_negative_prompt = (
    "stylized, cartoon, exaggerated anatomy, deformed, extra limbs, "
    "low quality, blurry, lowres"
)

prompt_additions = ", ".join(PROMPT_ADDITIONS) if "PROMPT_ADDITIONS" in globals() else ""
negative_additions = ", ".join(NEGATIVE_PROMPT_ADDITIONS) if "NEGATIVE_PROMPT_ADDITIONS" in globals() else ""

prompt = base_prompt + (", " + prompt_additions if prompt_additions else "")
negative_prompt = base_negative_prompt + (", " + negative_additions if negative_additions else "")

seeds = [1001, 1002, 1003, 1004]
num_steps = 30
guidance_scale = 5.5

controlnet_conditioning_images = []
for key in active_controlnet_keys:
    controlnet_conditioning_images.append(conditioning_by_key[key])

results = []
for seed in seeds:
    set_seed(seed)
    call_kwargs = {
        "prompt": prompt,
        "negative_prompt": negative_prompt,
        "num_inference_steps": num_steps,
        "guidance_scale": guidance_scale,
    }

    if controlnet_conditioning_images:
        call_kwargs["image"] = (
            controlnet_conditioning_images[0]
            if len(controlnet_conditioning_images) == 1
            else controlnet_conditioning_images
        )
        cond_w, cond_h = controlnet_conditioning_images[0].size
        call_kwargs["width"] = cond_w
        call_kwargs["height"] = cond_h

        scales = []
        for key in active_controlnet_keys:
            scale = CONTROLNET_CONDITIONING_SCALES.get(key, 1.0)
            scales.append(scale)
        call_kwargs["controlnet_conditioning_scale"] = scales[0] if len(scales) == 1 else scales
    else:
        call_kwargs["width"] = OUTPUT_WIDTH
        call_kwargs["height"] = OUTPUT_HEIGHT
        scales = []

    image = pipe(**call_kwargs).images[0]
    filename = f"candidate_seed_{seed}.png"
    out_path = EXPLORATION_DIR / filename
    image.save(out_path)
    results.append((seed, out_path, image))
    print(f"Saved {out_path}")

# Save generation metadata for reproducibility
meta = {
    "created_at": datetime.utcnow().isoformat() + "Z",
    "base_model_name": SELECTED_BASE_MODEL.name if "SELECTED_BASE_MODEL" in globals() else None,
    "base_model_path": str(SELECTED_BASE_MODEL.path) if "SELECTED_BASE_MODEL" in globals() else None,
    "base_family": BASE_MODEL_FAMILY if "BASE_MODEL_FAMILY" in globals() else None,
    "prompt": prompt,
    "negative_prompt": negative_prompt,
    "num_steps": num_steps,
    "guidance_scale": guidance_scale,
    "seeds": seeds,
    "width": call_kwargs.get("width"),
    "height": call_kwargs.get("height"),
    "use_controlnet": bool(controlnet_conditioning_images),
    "controlnet_keys": list(active_controlnet_keys),
    "controlnet_conditioning_scales": scales,
    "controlnet_source": str(source_path) if source_path else None,
}

meta_path = EXPLORATION_DIR / "generation_meta.json"
meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
print(f"Wrote {meta_path}")

if source_image is not None:
    for label, preview_img in [("Source", source_image)] + [(k.title(), v) for k, v in conditioning_by_key.items()]:
        print(f"
{label} preview")
        display(preview_img)

for seed, path, image in results:
    display(image)

PROMPT_ADDITIONS = []
NEGATIVE_PROMPT_ADDITIONS = []


: 

In [ ]:
from PIL import Image

if not results:
    print("No exploration results found.")
else:
    cols = 2
    rows = (len(results) + cols - 1) // cols
    w, h = results[0][2].size
    grid = Image.new("RGB", (w * cols, h * rows), (0, 0, 0))
    for idx, (_, _, img) in enumerate(results):
        r = idx // cols
        c = idx % cols
        grid.paste(img, (c * w, r * h))

    grid_path = EXPLORATION_DIR / "grid.png"
    grid.save(grid_path)
    display(grid)
    print(f"Saved {grid_path}")

: 

In [ ]:
import csv

csv_path = EXPLORATION_DIR / "candidates.csv"
with csv_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=[
            "seed",
            "image_path",
            "prompt",
            "negative_prompt",
            "num_steps",
            "guidance_scale"
        ]
    )
    writer.writeheader()
    for seed, path, _ in results:
        writer.writerow({
            "seed": seed,
            "image_path": str(path),
            "prompt": prompt,
            "negative_prompt": negative_prompt,
            "num_steps": num_steps,
            "guidance_scale": guidance_scale
        })

print(f"Wrote {csv_path}")

: 

---

# SECTION 4 — Candidate Selection & Identity Capture

**Purpose:**  
Promote one exploration image into a persistent character.

Steps:
- Select a single candidate image
- Assign a character name / ID
- Extract face embedding
- Capture base seed
- Initialize character directory

**Result:**  
A named character exists for the first time.


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import shutil
import csv

CHARACTER_ROOT = AI_DIRS["datasets"] / "characters"
CHARACTER_ROOT.mkdir(parents=True, exist_ok=True)


def latest_exploration_dir(root_dir: Path | None = None) -> Path:
    root_dir = root_dir or (AI_DIRS["images"] / "exploration")
    if not root_dir.exists():
        raise FileNotFoundError(f"Exploration root not found: {root_dir}")

    dirs = [p for p in root_dir.iterdir() if p.is_dir()]
    if not dirs:
        raise FileNotFoundError(f"No exploration runs found under {root_dir}")

    return sorted(dirs, key=lambda p: p.name, reverse=True)[0]


def get_runtime_versions():
    import importlib

    packages = [
        "torch",
        "torchvision",
        "xformers",
        "transformers",
        "diffusers",
        "accelerate",
        "bitsandbytes",
    ]
    versions = {}
    for package_name in packages:
        try:
            module = importlib.import_module(package_name)
            versions[package_name] = getattr(module, "__version__", "unknown")
        except ModuleNotFoundError:
            versions[package_name] = "not installed"
    return versions


def select_candidate_from_csv(seed_or_index, exploration_dir: Path | None = None):
    exploration_dir = exploration_dir or latest_exploration_dir()
    csv_path = exploration_dir / "candidates.csv"
    if not csv_path.exists():
        raise FileNotFoundError(f"Candidate CSV not found: {csv_path}")

    with csv_path.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))

    if not rows:
        raise ValueError(f"No candidates found in {csv_path}")

    # Try matching seed first, then fallback to index
    seed_str = str(seed_or_index)
    selected = None
    for row in rows:
        if str(row.get("seed")) == seed_str:
            selected = row
            break

    if selected is None:
        index = int(seed_or_index)
        if index < 0 or index >= len(rows):
            raise IndexError(f"Candidate index out of range: {index}")
        selected = rows[index]

    image_path = selected["image_path"]

    meta_path = exploration_dir / "generation_meta.json"
    generation_meta = {}
    if meta_path.exists():
        generation_meta = json.loads(meta_path.read_text(encoding="utf-8"))

    generation_meta.update({
        "selected_seed": int(selected.get("seed", 0)),
        "selected_image_path": image_path,
        "prompt": selected.get("prompt"),
        "negative_prompt": selected.get("negative_prompt"),
        "num_steps": int(selected.get("num_steps", 0)),
        "guidance_scale": float(selected.get("guidance_scale", 0)),
    })

    return image_path, generation_meta


def init_character_from_candidate(
    candidate_image_path: str,
    character_id: str,
    base_seed: int,
    notes: str = "",
    metadata: dict | None = None,
    face_embedding=None,
) -> Path:
    candidate_path = Path(candidate_image_path).expanduser()
    if not candidate_path.exists():
        raise FileNotFoundError(f"Candidate image not found: {candidate_path}")

    character_dir = CHARACTER_ROOT / character_id
    if character_dir.exists():
        raise FileExistsError(f"Character already exists: {character_dir}")
    character_dir.mkdir(parents=True, exist_ok=False)

    candidate_name = f"candidate{candidate_path.suffix}"
    candidate_dest = character_dir / candidate_name
    shutil.copy2(candidate_path, candidate_dest)

    embedding_status = "pending"
    embedding_dim = None
    embedding_data = None
    if face_embedding is not None:
        embedding_status = "captured"
        embedding_data = list(face_embedding)
        embedding_dim = len(embedding_data)

    generation_meta = metadata or {}
    if "runtime_versions" not in generation_meta:
        generation_meta["runtime_versions"] = get_runtime_versions()

    metadata_out = {
        "character_id": character_id,
        "created_at": datetime.utcnow().isoformat() + "Z",
        "candidate_image": candidate_name,
        "base_seed": int(base_seed),
        "notes": notes,
        "embedding_status": embedding_status,
        "embedding_dim": embedding_dim,
        "embedding": embedding_data,
        "generation": generation_meta,
    }

    (character_dir / "character.json").write_text(
        json.dumps(metadata_out, indent=2),
        encoding="utf-8",
    )
    (character_dir / "seed.txt").write_text(str(base_seed), encoding="utf-8")

    print(f"Initialized character at {character_dir}")
    return character_dir


: 

In [ ]:
# Option A: Set candidate_image_path manually
candidate_image_path = ""  # e.g., /content/drive/My Drive/AI/Images/exploration/20240101_000000/candidate_seed_1001.png
candidate_metadata = None

# Option B: Pick from latest exploration by seed or index
# candidate_image_path, candidate_metadata = select_candidate_from_csv(1001)  # by seed
# candidate_image_path, candidate_metadata = select_candidate_from_csv(0)     # by index

character_id = "CH-0001"
base_seed = 0
notes = ""

if not candidate_image_path:
    print("Set candidate_image_path or use select_candidate_from_csv and re-run this cell.")
else:
    init_character_from_candidate(
        candidate_image_path=candidate_image_path,
        character_id=character_id,
        base_seed=base_seed,
        notes=notes,
        metadata=candidate_metadata,
    )


: 

---

# SECTION 5 — Identity Locking & Invariant Feature Definition

**Purpose:**  
Define what can never change for this character.

Includes:
- Facial geometry lock
- Body proportion constraints
- Invariant skin feature masks
- Identity validation checks

From this point forward, the character is identity-locked.


In [ ]:
# SECTION 5 — Identity Locking & Invariant Feature Definition (Core Implementation)
import hashlib
from typing import Any

import numpy as np
from PIL import Image

IDENTITY_LOCK_CONFIG = {
    "require_face_detection": False,
    "fail_on_missing_mask": True,
    "insightface_threshold_centroid_min": 0.35,
    "insightface_threshold_best_ref_min": 0.45,
    "clip_threshold_centroid_min": 0.82,
    "clip_threshold_best_ref_min": 0.86,
    "phash_threshold_hamming_max": 10,
    "clip_model_id": "openai/clip-vit-base-patch32",
}

_IDENTITY_BACKEND_CACHE = {
    "insightface_app": None,
    "clip_processor": None,
    "clip_model": None,
    "clip_device": "cpu",
}


def _utc_now_iso() -> str:
    return datetime.utcnow().isoformat() + "Z"


def _character_dir(character_id: str) -> Path:
    character_dir = CHARACTER_ROOT / character_id
    if not character_dir.exists():
        raise FileNotFoundError(f"Character not found: {character_dir}")
    return character_dir


def _character_record_path(character_id: str) -> Path:
    return _character_dir(character_id) / "character.json"


def _sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def _safe_resample_lanczos():
    return Image.Resampling.LANCZOS if hasattr(Image, "Resampling") else Image.LANCZOS


def _normalize_vector(vector) -> np.ndarray | None:
    arr = np.asarray(vector, dtype=np.float32).reshape(-1)
    if arr.size == 0:
        return None
    norm = float(np.linalg.norm(arr))
    if norm <= 0.0:
        return None
    return (arr / norm).astype(np.float32)


def _vector_centroid(vectors: list) -> np.ndarray | None:
    normalized = []
    for vector in vectors:
        n = _normalize_vector(vector)
        if n is not None:
            normalized.append(n)
    if not normalized:
        return None
    stacked = np.vstack(normalized)
    return _normalize_vector(stacked.mean(axis=0))


def _cosine_similarity(vec_a, vec_b) -> float | None:
    a = _normalize_vector(vec_a)
    b = _normalize_vector(vec_b)
    if a is None or b is None or a.shape != b.shape:
        return None
    return float(np.dot(a, b))


def _copy_to_character_subdir(
    character_dir: Path,
    source_path: str | Path,
    subdir: str,
    target_name: str,
) -> tuple[str, str]:
    src = Path(source_path).expanduser()
    if not src.exists():
        raise FileNotFoundError(f"Source file not found: {src}")

    dst_dir = character_dir / subdir
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / target_name
    shutil.copy2(src, dst)
    return dst.relative_to(character_dir).as_posix(), _sha256_file(dst)


def _load_image_rgb(path: Path) -> Image.Image:
    with Image.open(path) as handle:
        return handle.convert("RGB")


def _load_insightface_app() -> tuple[Any | None, list[str]]:
    warnings = []
    cached = _IDENTITY_BACKEND_CACHE.get("insightface_app")
    if cached is not None:
        return cached, warnings

    try:
        from insightface.app import FaceAnalysis
    except Exception as exc:
        warnings.append(f"InsightFace import failed: {exc}")
        return None, warnings

    provider_attempts = [
        (["CUDAExecutionProvider", "CPUExecutionProvider"], 0),
        (["CPUExecutionProvider"], -1),
        (None, -1),
    ]
    for providers, ctx_id in provider_attempts:
        try:
            kwargs = {"name": "buffalo_l"}
            if providers is not None:
                kwargs["providers"] = providers
            app = FaceAnalysis(**kwargs)
            app.prepare(ctx_id=ctx_id, det_size=(640, 640))
            _IDENTITY_BACKEND_CACHE["insightface_app"] = app
            return app, warnings
        except Exception as exc:
            label = ",".join(providers) if providers else "default"
            warnings.append(f"InsightFace init failed ({label}): {exc}")

    return None, warnings


def _extract_insightface_embedding(image: Image.Image) -> tuple[np.ndarray | None, list[str]]:
    app, warnings = _load_insightface_app()
    if app is None:
        return None, warnings

    rgb = np.asarray(image.convert("RGB"))
    if rgb.ndim != 3 or rgb.shape[2] != 3:
        warnings.append("InsightFace skipped due to invalid image format.")
        return None, warnings
    bgr = rgb[:, :, ::-1]

    try:
        faces = app.get(bgr)
    except Exception as exc:
        warnings.append(f"InsightFace face detection failed: {exc}")
        return None, warnings

    if not faces:
        warnings.append("InsightFace found no face in image.")
        return None, warnings

    def _face_area(face_obj):
        try:
            x0, y0, x1, y1 = face_obj.bbox
            return float(max(0.0, x1 - x0) * max(0.0, y1 - y0))
        except Exception:
            return 0.0

    face = max(faces, key=_face_area)
    emb = getattr(face, "normed_embedding", None)
    if emb is None:
        emb = getattr(face, "embedding", None)
    vec = _normalize_vector(emb)
    if vec is None:
        warnings.append("InsightFace returned an invalid embedding.")
        return None, warnings
    return vec, warnings


def _load_clip_stack() -> tuple[tuple[Any, Any, str] | None, list[str]]:
    warnings = []
    processor = _IDENTITY_BACKEND_CACHE.get("clip_processor")
    model = _IDENTITY_BACKEND_CACHE.get("clip_model")
    device = _IDENTITY_BACKEND_CACHE.get("clip_device", "cpu")
    if processor is not None and model is not None:
        return (processor, model, device), warnings

    try:
        import torch
        from transformers import CLIPModel, CLIPProcessor
    except Exception as exc:
        warnings.append(f"CLIP import failed: {exc}")
        return None, warnings

    model_id = IDENTITY_LOCK_CONFIG["clip_model_id"]
    try:
        processor = CLIPProcessor.from_pretrained(model_id)
        model = CLIPModel.from_pretrained(model_id)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model = model.to(device)
        model.eval()
        _IDENTITY_BACKEND_CACHE["clip_processor"] = processor
        _IDENTITY_BACKEND_CACHE["clip_model"] = model
        _IDENTITY_BACKEND_CACHE["clip_device"] = device
        return (processor, model, device), warnings
    except Exception as exc:
        warnings.append(f"CLIP model load failed ({model_id}): {exc}")
        return None, warnings


def _extract_clip_embedding(image: Image.Image) -> tuple[np.ndarray | None, list[str]]:
    stack, warnings = _load_clip_stack()
    if stack is None:
        return None, warnings

    processor, model, device = stack
    try:
        import torch

        inputs = processor(images=image.convert("RGB"), return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            features = model.get_image_features(**inputs)
        vec = _normalize_vector(features[0].detach().cpu().numpy())
    except Exception as exc:
        warnings.append(f"CLIP embedding failed: {exc}")
        return None, warnings

    if vec is None:
        warnings.append("CLIP returned an invalid embedding.")
        return None, warnings
    return vec, warnings


def _dhash_hex(image: Image.Image, hash_size: int = 8) -> str:
    resized = image.convert("L").resize((hash_size + 1, hash_size), _safe_resample_lanczos())
    pixels = np.asarray(resized, dtype=np.int16)
    diff = pixels[:, 1:] > pixels[:, :-1]
    bit_string = "".join("1" if v else "0" for v in diff.flatten())
    width = (hash_size * hash_size) // 4
    return f"{int(bit_string, 2):0{width}x}"


def _extract_phash_hex(image: Image.Image) -> tuple[str | None, list[str]]:
    warnings = []
    try:
        import imagehash

        return str(imagehash.phash(image.convert("RGB"))), warnings
    except Exception as exc:
        warnings.append(f"imagehash unavailable ({exc}); using dhash fallback.")
        try:
            return _dhash_hex(image), warnings
        except Exception as inner_exc:
            warnings.append(f"dhash fallback failed: {inner_exc}")
            return None, warnings


def _hamming_distance_hex(hex_a: str, hex_b: str) -> int:
    a = int(hex_a, 16)
    b = int(hex_b, 16)
    return int((a ^ b).bit_count())


def _collect_reference_fingerprints(reference_images: list[Image.Image]) -> tuple[dict, list[str]]:
    warnings = []
    insightface_vectors = []
    clip_vectors = []
    phash_values = []

    for index, image in enumerate(reference_images):
        i_vec, i_warn = _extract_insightface_embedding(image)
        warnings.extend([f"reference[{index}] {msg}" for msg in i_warn])
        if i_vec is not None:
            insightface_vectors.append(i_vec.tolist())

        c_vec, c_warn = _extract_clip_embedding(image)
        warnings.extend([f"reference[{index}] {msg}" for msg in c_warn])
        if c_vec is not None:
            clip_vectors.append(c_vec.tolist())

        p_hash, p_warn = _extract_phash_hex(image)
        warnings.extend([f"reference[{index}] {msg}" for msg in p_warn])
        if p_hash is not None:
            phash_values.append(p_hash)

    insightface_centroid = _vector_centroid(insightface_vectors)
    clip_centroid = _vector_centroid(clip_vectors)

    fingerprints = {
        "backend_used": None,
        "insightface": {
            "available": bool(insightface_vectors),
            "embedding_dim": len(insightface_vectors[0]) if insightface_vectors else 512,
            "reference_embeddings": insightface_vectors,
            "centroid": insightface_centroid.tolist() if insightface_centroid is not None else [],
            "threshold_centroid_min": float(IDENTITY_LOCK_CONFIG["insightface_threshold_centroid_min"]),
            "threshold_best_ref_min": float(IDENTITY_LOCK_CONFIG["insightface_threshold_best_ref_min"]),
        },
        "clip": {
            "available": bool(clip_vectors),
            "embedding_dim": len(clip_vectors[0]) if clip_vectors else 1024,
            "reference_embeddings": clip_vectors,
            "centroid": clip_centroid.tolist() if clip_centroid is not None else [],
            "threshold_centroid_min": float(IDENTITY_LOCK_CONFIG["clip_threshold_centroid_min"]),
            "threshold_best_ref_min": float(IDENTITY_LOCK_CONFIG["clip_threshold_best_ref_min"]),
        },
        "phash": {
            "available": bool(phash_values),
            "reference_hashes": phash_values,
            "threshold_hamming_max": int(IDENTITY_LOCK_CONFIG["phash_threshold_hamming_max"]),
        },
    }

    if fingerprints["insightface"]["available"]:
        fingerprints["backend_used"] = "insightface"
    elif fingerprints["clip"]["available"]:
        fingerprints["backend_used"] = "clip"
    elif fingerprints["phash"]["available"]:
        fingerprints["backend_used"] = "phash"

    return fingerprints, warnings


def _extract_candidate_fingerprints(candidate_image: Image.Image) -> tuple[dict, list[str]]:
    warnings = []
    i_vec, i_warn = _extract_insightface_embedding(candidate_image)
    warnings.extend(i_warn)
    c_vec, c_warn = _extract_clip_embedding(candidate_image)
    warnings.extend(c_warn)
    p_hash, p_warn = _extract_phash_hex(candidate_image)
    warnings.extend(p_warn)

    return {
        "insightface": i_vec.tolist() if i_vec is not None else None,
        "clip": c_vec.tolist() if c_vec is not None else None,
        "phash": p_hash,
    }, warnings


def load_character_record(character_id: str) -> dict:
    record_path = _character_record_path(character_id)
    if not record_path.exists():
        raise FileNotFoundError(f"Character record not found: {record_path}")
    with record_path.open("r", encoding="utf-8") as handle:
        record = json.load(handle)
    if not isinstance(record, dict):
        raise ValueError(f"Character record is not a dict: {record_path}")
    return record


def save_character_record(character_id: str, record: dict) -> Path:
    if not isinstance(record, dict):
        raise TypeError("record must be a dictionary")
    record_path = _character_record_path(character_id)
    record_path.write_text(json.dumps(record, indent=2), encoding="utf-8")
    return record_path


def create_identity_lock(
    character_id: str,
    reference_image_paths: list[str],
    invariant_features: list[dict],
    locked_regions: list[str],
    notes: str = "",
) -> dict:
    if not reference_image_paths:
        raise ValueError("reference_image_paths must include at least one image")

    character_dir = _character_dir(character_id)
    record = load_character_record(character_id)

    reference_entries = []
    reference_images = []
    for index, source in enumerate(reference_image_paths, start=1):
        src_path = Path(source).expanduser()
        ext = src_path.suffix.lower() or ".png"
        target_name = f"ref_{index:03d}{ext}"
        rel_path, sha256 = _copy_to_character_subdir(
            character_dir,
            src_path,
            "identity/reference",
            target_name,
        )
        reference_entries.append({"file": rel_path, "sha256": sha256})
        reference_images.append(_load_image_rgb(character_dir / rel_path))

    invariant_entries = []
    for index, feature in enumerate(invariant_features or [], start=1):
        if not isinstance(feature, dict):
            raise TypeError(f"Invariant feature at index {index - 1} must be a dict")
        feature_id = str(feature.get("id") or f"INV-{index:03d}")
        kind = str(feature.get("kind") or "other")
        region = str(feature.get("region") or "unspecified")
        description = str(feature.get("description") or "")

        mask_source = feature.get("mask_path")
        mask_rel = None
        mask_sha = None
        if mask_source:
            mask_source_path = Path(str(mask_source)).expanduser()
            mask_ext = mask_source_path.suffix.lower() or ".png"
            mask_name = f"{feature_id.lower().replace(' ', '_')}{mask_ext}"
            mask_rel, mask_sha = _copy_to_character_subdir(
                character_dir,
                mask_source_path,
                "identity/masks",
                mask_name,
            )

        invariant_entries.append({
            "id": feature_id,
            "kind": kind,
            "region": region,
            "description": description,
            "mask_file": mask_rel,
            "mask_sha256": mask_sha,
        })

    fingerprints, fingerprint_warnings = _collect_reference_fingerprints(reference_images)
    if fingerprints.get("backend_used") is None:
        raise RuntimeError("Unable to compute any fingerprint backend for identity lock.")

    if IDENTITY_LOCK_CONFIG["require_face_detection"] and not fingerprints["insightface"]["available"]:
        raise RuntimeError("InsightFace face detection is required but no face embedding was produced.")

    identity_lock = {
        "status": "locked",
        "locked_at": _utc_now_iso(),
        "locked_regions": list(locked_regions or []),
        "notes": str(notes or ""),
        "reference_images": reference_entries,
        "invariants": {
            "features": invariant_entries,
        },
        "fingerprints": fingerprints,
        "last_validation": None,
    }
    if fingerprint_warnings:
        identity_lock["warnings"] = fingerprint_warnings

    record["identity_lock"] = identity_lock
    save_character_record(character_id, record)
    print(f"Identity lock created for {character_id} using backend={fingerprints['backend_used']}.")
    return identity_lock


def validate_identity_lock(character_id: str, candidate_image_path: str) -> dict:
    candidate_path = Path(candidate_image_path).expanduser()
    if not candidate_path.exists():
        raise FileNotFoundError(f"Candidate image not found: {candidate_path}")

    record = load_character_record(character_id)
    identity_lock = record.get("identity_lock")
    if not isinstance(identity_lock, dict):
        raise ValueError(f"Character {character_id} does not contain an identity_lock block.")

    stored_fingerprints = identity_lock.get("fingerprints", {})
    candidate_image = _load_image_rgb(candidate_path)
    candidate_fingerprints, candidate_warnings = _extract_candidate_fingerprints(candidate_image)

    backend_used = None
    backend_pass = False
    metrics = {}
    reasons = []

    stored_insight = stored_fingerprints.get("insightface", {})
    stored_clip = stored_fingerprints.get("clip", {})
    stored_phash = stored_fingerprints.get("phash", {})

    if stored_insight.get("available") and candidate_fingerprints.get("insightface") is not None:
        backend_used = "insightface"
        candidate_vec = candidate_fingerprints["insightface"]
        centroid = stored_insight.get("centroid") or []
        ref_vectors = stored_insight.get("reference_embeddings") or []
        centroid_sim = _cosine_similarity(candidate_vec, centroid) if centroid else None
        best_ref_sim = None
        if ref_vectors:
            similarities = [_cosine_similarity(candidate_vec, ref) for ref in ref_vectors]
            similarities = [s for s in similarities if s is not None]
            best_ref_sim = max(similarities) if similarities else None

        threshold_centroid = float(stored_insight.get(
            "threshold_centroid_min",
            IDENTITY_LOCK_CONFIG["insightface_threshold_centroid_min"],
        ))
        threshold_best_ref = float(stored_insight.get(
            "threshold_best_ref_min",
            IDENTITY_LOCK_CONFIG["insightface_threshold_best_ref_min"],
        ))
        centroid_ok = centroid_sim is not None and centroid_sim >= threshold_centroid
        best_ref_ok = best_ref_sim is not None and best_ref_sim >= threshold_best_ref
        backend_pass = centroid_ok and best_ref_ok
        metrics.update({
            "similarity_centroid": centroid_sim,
            "similarity_best_ref": best_ref_sim,
            "threshold_centroid_min": threshold_centroid,
            "threshold_best_ref_min": threshold_best_ref,
            "centroid_pass": centroid_ok,
            "best_ref_pass": best_ref_ok,
        })
        if not centroid_ok:
            reasons.append("InsightFace centroid similarity below threshold.")
        if not best_ref_ok:
            reasons.append("InsightFace best-reference similarity below threshold.")
    elif stored_clip.get("available") and candidate_fingerprints.get("clip") is not None:
        backend_used = "clip"
        candidate_vec = candidate_fingerprints["clip"]
        centroid = stored_clip.get("centroid") or []
        ref_vectors = stored_clip.get("reference_embeddings") or []
        centroid_sim = _cosine_similarity(candidate_vec, centroid) if centroid else None
        best_ref_sim = None
        if ref_vectors:
            similarities = [_cosine_similarity(candidate_vec, ref) for ref in ref_vectors]
            similarities = [s for s in similarities if s is not None]
            best_ref_sim = max(similarities) if similarities else None

        threshold_centroid = float(stored_clip.get(
            "threshold_centroid_min",
            IDENTITY_LOCK_CONFIG["clip_threshold_centroid_min"],
        ))
        threshold_best_ref = float(stored_clip.get(
            "threshold_best_ref_min",
            IDENTITY_LOCK_CONFIG["clip_threshold_best_ref_min"],
        ))
        centroid_ok = centroid_sim is not None and centroid_sim >= threshold_centroid
        best_ref_ok = best_ref_sim is not None and best_ref_sim >= threshold_best_ref
        backend_pass = centroid_ok and best_ref_ok
        metrics.update({
            "similarity_centroid": centroid_sim,
            "similarity_best_ref": best_ref_sim,
            "threshold_centroid_min": threshold_centroid,
            "threshold_best_ref_min": threshold_best_ref,
            "centroid_pass": centroid_ok,
            "best_ref_pass": best_ref_ok,
        })
        if not centroid_ok:
            reasons.append("CLIP centroid similarity below threshold.")
        if not best_ref_ok:
            reasons.append("CLIP best-reference similarity below threshold.")
    elif stored_phash.get("available") and candidate_fingerprints.get("phash"):
        backend_used = "phash"
        candidate_hash = candidate_fingerprints["phash"]
        ref_hashes = stored_phash.get("reference_hashes") or []
        threshold_max = int(stored_phash.get(
            "threshold_hamming_max",
            IDENTITY_LOCK_CONFIG["phash_threshold_hamming_max"],
        ))
        if ref_hashes:
            distances = [_hamming_distance_hex(candidate_hash, ref_hash) for ref_hash in ref_hashes]
            best_distance = min(distances)
        else:
            best_distance = None
        backend_pass = best_distance is not None and best_distance <= threshold_max
        metrics.update({
            "hamming_best_ref": best_distance,
            "threshold_hamming_max": threshold_max,
        })
        if not backend_pass:
            reasons.append("pHash hamming distance above threshold.")
    else:
        reasons.append("No compatible fingerprint backend available for validation.")

    mask_pass = True
    missing_masks = []
    mismatched_masks = []
    for feature in identity_lock.get("invariants", {}).get("features", []):
        if not isinstance(feature, dict):
            continue
        mask_rel = feature.get("mask_file")
        expected_sha = feature.get("mask_sha256")
        if not mask_rel:
            continue
        mask_path = _character_dir(character_id) / mask_rel
        if not mask_path.exists():
            missing_masks.append(mask_rel)
            if IDENTITY_LOCK_CONFIG["fail_on_missing_mask"]:
                mask_pass = False
            continue
        if expected_sha:
            actual_sha = _sha256_file(mask_path)
            if actual_sha != expected_sha:
                mismatched_masks.append(mask_rel)
                if IDENTITY_LOCK_CONFIG["fail_on_missing_mask"]:
                    mask_pass = False

    if missing_masks:
        reasons.append(f"Missing invariant mask files: {missing_masks}")
    if mismatched_masks:
        reasons.append(f"Invariant mask hash mismatches: {mismatched_masks}")
    if IDENTITY_LOCK_CONFIG["require_face_detection"] and backend_used != "insightface":
        reasons.append("Face detection required, but validation used a fallback backend.")
        backend_pass = False

    validation_pass = backend_pass and mask_pass
    validation_report = {
        "timestamp": _utc_now_iso(),
        "pass": validation_pass,
        "backend_used": backend_used,
        "metrics": metrics,
        "candidate_image": str(candidate_path),
        "warnings": candidate_warnings,
        "reasons": reasons,
    }

    identity_lock["last_validation"] = validation_report
    record["identity_lock"] = identity_lock
    save_character_record(character_id, record)

    print(json.dumps(validation_report, indent=2))
    return validation_report


In [ ]:
print("Section 5 identity lock configuration:")
print(json.dumps(IDENTITY_LOCK_CONFIG, indent=2))

# Section 5 usage example — create lock
identity_lock_character_id = "CH-0001"
identity_reference_image_paths = []  # e.g., ["/content/drive/My Drive/AI/Images/identity/front.png"]
identity_invariant_features = [
    # {"id": "INV-001", "kind": "scar", "region": "left_cheek", "description": "Small vertical scar", "mask_path": "/path/to/mask.png"},
]
identity_locked_regions = ["face", "torso"]
identity_lock_notes = ""

if identity_reference_image_paths:
    created_identity_lock = create_identity_lock(
        character_id=identity_lock_character_id,
        reference_image_paths=identity_reference_image_paths,
        invariant_features=identity_invariant_features,
        locked_regions=identity_locked_regions,
        notes=identity_lock_notes,
    )
    print(f"Created identity lock with backend: {created_identity_lock['fingerprints']['backend_used']}")
else:
    print("Set identity_reference_image_paths to create an identity lock.")

# Section 5 usage example — validate a candidate
validation_character_id = identity_lock_character_id
validation_candidate_image_path = ""  # e.g., /content/drive/My Drive/AI/Images/renders/candidate.png

if validation_candidate_image_path:
    validation_result = validate_identity_lock(
        character_id=validation_character_id,
        candidate_image_path=validation_candidate_image_path,
    )
    print(f"Validation pass: {validation_result['pass']}")
else:
    print("Set validation_candidate_image_path to run identity validation.")


---

# SECTION 6 — Body Region Map Creation (Distinctive Anchor: S6_BODY_REGION_MAP_SCHEMA_V1)

**Purpose:**  
Divide the body into anatomically meaningful regions for refinement.

Example regions:
- Head
- Neck
- Shoulders
- Chest
- Abdomen
- Hips
- Thighs
- Calves
- Upper arms
- Forearms
- Hands
- Feet

Each region receives a mask or index.


In [ ]:
# SECTION 6 — Body Region Map Creation (Distinctive Anchor: S6_BODY_REGION_MAP_SCHEMA_V1)
SECTION6_BODY_REGION_SCHEMA_VERSION = "section6.body_region_map.v1"
SECTION6_READY_MESSAGE = "Section 6 complete; safe to begin Section 7."
SECTION6_REGION_DEFAULTS = [
    ("head", "Head", "none", "core"),
    ("neck", "Neck", "none", "core"),
    ("shoulders", "Shoulders", "bilateral", "upper_body"),
    ("chest", "Chest", "none", "torso"),
    ("abdomen", "Abdomen", "none", "torso"),
    ("hips", "Hips", "bilateral", "pelvis"),
    ("thighs", "Thighs", "bilateral", "legs"),
    ("calves", "Calves", "bilateral", "legs"),
    ("upper_arms", "Upper Arms", "bilateral", "arms"),
    ("forearms", "Forearms", "bilateral", "arms"),
    ("hands", "Hands", "bilateral", "arms"),
    ("feet", "Feet", "bilateral", "legs"),
]
SECTION6_REQUIRED_REGION_KEYS = [region_id for region_id, _, _, _ in SECTION6_REGION_DEFAULTS]
SECTION6_REQUIRED_MASK_REGIONS = list(SECTION6_REQUIRED_REGION_KEYS)
SECTION6_REGION_MASK_SUBDIR = "identity/masks/regions"


def _section6_region_slug(region_id: str) -> str:
    slug = "".join(ch.lower() if (ch.isalnum() or ch in {"_", "-"}) else "_" for ch in str(region_id).strip())
    slug = "_".join(filter(None, slug.replace("-", "_").split("_")))
    if not slug:
        raise ValueError(f"Invalid region_id for mask path resolution: {region_id!r}")
    return slug


def _section6_region_mask_target_name(region_id: str, source_path: str | Path | None = None) -> str:
    ext = ".png"
    if source_path is not None:
        ext = Path(source_path).suffix.lower() or ".png"
    return f"{_section6_region_slug(region_id)}{ext}"


def _section6_region_mask_relpath(region_id: str, source_path: str | Path | None = None) -> str:
    return f"{SECTION6_REGION_MASK_SUBDIR}/{_section6_region_mask_target_name(region_id, source_path)}"


def _read_mask_metadata(mask_path: Path) -> dict:
    metadata = {
        "exists": mask_path.exists(),
        "relative_path": mask_path.as_posix(),
    }
    if not mask_path.exists():
        return metadata

    metadata["sha256"] = _sha256_file(mask_path)
    try:
        with Image.open(mask_path) as image:
            metadata["format"] = image.format
            metadata["mode"] = image.mode
            metadata["width"] = image.width
            metadata["height"] = image.height
    except Exception as exc:
        metadata["warning"] = f"Unable to parse mask image metadata: {exc}"
    return metadata


def _default_body_region_entry(
    region_id: str,
    label: str,
    side: str,
    parent_group: str,
    *,
    operation_tag: str,
) -> dict:
    return {
        "region_id": region_id,
        "label": label,
        "side": side,
        "parent_group": parent_group,
        "mask_path": _section6_region_mask_relpath(region_id),
        "mask_metadata": None,
        "is_locked": False,
        "is_frozen": False,
        "last_edited_at": _utc_now_iso(),
        "source_operation_tag": operation_tag,
        "notes": None,
    }


def _normalize_body_region_entry(region_id: str, region_payload: dict, fallback_defaults: tuple[str, str, str], *, operation_tag: str) -> dict:
    label, side, parent_group = fallback_defaults
    payload = dict(region_payload) if isinstance(region_payload, dict) else {}

    payload.setdefault("region_id", region_id)
    payload.setdefault("label", label)
    payload.setdefault("side", side)
    payload.setdefault("parent_group", parent_group)
    payload.setdefault("mask_path", _section6_region_mask_relpath(region_id))
    payload.setdefault("mask_metadata", None)
    payload["is_locked"] = bool(payload.get("is_locked", False))
    payload["is_frozen"] = bool(payload.get("is_frozen", False))
    payload.setdefault("last_edited_at", _utc_now_iso())
    payload.setdefault("source_operation_tag", operation_tag)
    payload.setdefault("notes", None)
    return payload


def build_default_body_region_map(*, source: str = "section6_initializer", pose_reference: str | None = None, image_reference: str | None = None) -> dict:
    regions = {
        region_id: _default_body_region_entry(region_id, label, side, parent_group, operation_tag=source)
        for region_id, label, side, parent_group in SECTION6_REGION_DEFAULTS
    }
    return {
        "schema_version": SECTION6_BODY_REGION_SCHEMA_VERSION,
        "created_at": _utc_now_iso(),
        "updated_at": _utc_now_iso(),
        "source": source,
        "pose_reference": pose_reference,
        "image_reference": image_reference,
        "mask_root": SECTION6_REGION_MASK_SUBDIR,
        "section7_handoff": {
            "ready": False,
            "message": None,
            "finalized_at": None,
            "required_region_set": list(SECTION6_REQUIRED_REGION_KEYS),
        },
        "regions": regions,
    }


def ensure_body_region_map(character_id: str, *, source: str = "section6_initializer", pose_reference: str | None = None, image_reference: str | None = None) -> dict:
    """Backward-safe initializer for Section 6 payload under character.json."""
    record = load_character_record(character_id)
    existing_map = record.get("body_region_map")

    if not isinstance(existing_map, dict) or not isinstance(existing_map.get("regions"), dict):
        body_region_map = build_default_body_region_map(
            source=source,
            pose_reference=pose_reference,
            image_reference=image_reference,
        )
        record["body_region_map"] = body_region_map
        save_character_record(character_id, record)
        return body_region_map

    existing_map.setdefault("mask_root", SECTION6_REGION_MASK_SUBDIR)
    existing_map.setdefault("source", source)
    existing_map.setdefault("pose_reference", pose_reference)
    existing_map.setdefault("image_reference", image_reference)
    existing_map.setdefault("created_at", _utc_now_iso())
    existing_map["updated_at"] = _utc_now_iso()

    handoff = existing_map.get("section7_handoff") if isinstance(existing_map.get("section7_handoff"), dict) else {}
    handoff.setdefault("ready", False)
    handoff.setdefault("message", None)
    handoff.setdefault("finalized_at", None)
    handoff.setdefault("required_region_set", list(SECTION6_REQUIRED_REGION_KEYS))
    existing_map["section7_handoff"] = handoff

    defaults_lookup = {region_id: (label, side, parent_group) for region_id, label, side, parent_group in SECTION6_REGION_DEFAULTS}
    regions = existing_map.get("regions", {})
    for region_id in SECTION6_REQUIRED_REGION_KEYS:
        defaults = defaults_lookup[region_id]
        regions[region_id] = _normalize_body_region_entry(
            region_id,
            regions.get(region_id, {}),
            defaults,
            operation_tag=source,
        )
    existing_map["regions"] = regions

    record["body_region_map"] = existing_map
    save_character_record(character_id, record)
    return existing_map


def initialize_body_region_map(
    character_id: str,
    *,
    reset_existing: bool = False,
    source: str = "section6_initializer",
    pose_reference: str | None = None,
    image_reference: str | None = None,
) -> dict:
    record = load_character_record(character_id)
    existing_map = record.get("body_region_map")

    if reset_existing or not isinstance(existing_map, dict):
        body_region_map = build_default_body_region_map(
            source=source,
            pose_reference=pose_reference,
            image_reference=image_reference,
        )
        record["body_region_map"] = body_region_map
        save_character_record(character_id, record)
        return body_region_map

    return ensure_body_region_map(
        character_id,
        source=source,
        pose_reference=pose_reference,
        image_reference=image_reference,
    )


def register_body_region_mask_paths(
    character_id: str,
    region_masks: dict[str, str],
    *,
    validate_existing: bool = True,
    source_operation_tag: str = "section6.register_mask_paths",
) -> dict:
    record = load_character_record(character_id)
    body_region_map = ensure_body_region_map(character_id)
    regions = body_region_map.get("regions", {})

    for region_id, mask_path in region_masks.items():
        if region_id not in regions:
            raise KeyError(f"Unknown region_id for Section 6 registration: {region_id}")

        if not isinstance(mask_path, str) or not mask_path.strip():
            raise ValueError(f"mask_path must be a non-empty string for region '{region_id}'.")

        resolved_mask_path = Path(mask_path).expanduser()
        if validate_existing and not resolved_mask_path.exists():
            raise FileNotFoundError(f"Mask file not found for region '{region_id}': {resolved_mask_path}")

        region_payload = regions.get(region_id, {})
        if not isinstance(region_payload, dict):
            defaults = next((label, side, parent_group) for key, label, side, parent_group in SECTION6_REGION_DEFAULTS if key == region_id)
            region_payload = _default_body_region_entry(region_id, *defaults, operation_tag=source_operation_tag)

        region_payload["region_id"] = region_id
        region_payload["mask_path"] = str(resolved_mask_path)
        region_payload["mask_metadata"] = _read_mask_metadata(resolved_mask_path)
        region_payload["last_edited_at"] = _utc_now_iso()
        region_payload["source_operation_tag"] = source_operation_tag
        regions[region_id] = region_payload

    body_region_map["regions"] = regions
    body_region_map["updated_at"] = _utc_now_iso()
    body_region_map["section7_handoff"] = {
        "ready": False,
        "message": None,
        "finalized_at": None,
        "required_region_set": list(SECTION6_REQUIRED_REGION_KEYS),
    }
    record["body_region_map"] = body_region_map
    save_character_record(character_id, record)
    return body_region_map


def _validate_body_region_map_payload(body_region_map: dict) -> dict:
    errors = []
    warnings = []
    details = {
        "required_region_keys": list(SECTION6_REQUIRED_REGION_KEYS),
        "required_mask_regions": list(SECTION6_REQUIRED_MASK_REGIONS),
    }

    if not isinstance(body_region_map, dict):
        return {
            "pass": False,
            "errors": ["body_region_map must be a dictionary."],
            "warnings": [],
            "details": details,
        }

    regions = body_region_map.get("regions")
    if not isinstance(regions, dict):
        return {
            "pass": False,
            "errors": ["body_region_map.regions must be a dictionary."],
            "warnings": [],
            "details": details,
        }

    missing_required_keys = sorted(set(SECTION6_REQUIRED_REGION_KEYS) - set(regions.keys()))
    if missing_required_keys:
        errors.append(f"Missing required region keys: {missing_required_keys}")

    missing_required_masks = []
    invalid_mask_paths = []
    missing_provenance_fields = []
    invalid_freeze_flags = []
    for region_id in SECTION6_REQUIRED_REGION_KEYS:
        payload = regions.get(region_id)
        if not isinstance(payload, dict):
            continue

        mask_path = payload.get("mask_path")
        if not isinstance(mask_path, str) or not mask_path.strip():
            missing_required_masks.append(region_id)
        elif not Path(mask_path).expanduser().exists():
            invalid_mask_paths.append({"region_id": region_id, "mask_path": mask_path})

        for provenance_key in ("last_edited_at", "source_operation_tag"):
            if not isinstance(payload.get(provenance_key), str) or not payload.get(provenance_key):
                missing_provenance_fields.append({"region_id": region_id, "field": provenance_key})

        for freeze_flag in ("is_locked", "is_frozen"):
            if not isinstance(payload.get(freeze_flag), bool):
                invalid_freeze_flags.append({"region_id": region_id, "field": freeze_flag, "value": payload.get(freeze_flag)})

    if missing_required_masks:
        errors.append(f"Missing mask_path for required regions: {sorted(missing_required_masks)}")
    if invalid_mask_paths:
        errors.append(f"Mask files not found on disk: {invalid_mask_paths}")
    if missing_provenance_fields:
        errors.append(f"Missing Section 7 provenance fields: {missing_provenance_fields}")
    if invalid_freeze_flags:
        errors.append(f"Invalid freeze/lock flags: {invalid_freeze_flags}")

    details.update({
        "schema_version": body_region_map.get("schema_version"),
        "region_count": len(regions),
        "missing_required_region_keys": missing_required_keys,
        "missing_required_masks": sorted(missing_required_masks),
        "invalid_mask_paths": invalid_mask_paths,
        "missing_provenance_fields": missing_provenance_fields,
        "invalid_freeze_flags": invalid_freeze_flags,
    })

    return {
        "pass": len(errors) == 0,
        "errors": errors,
        "warnings": warnings,
        "details": details,
    }


def validate_body_region_map(character_id: str) -> dict:
    record = load_character_record(character_id)
    body_region_map = record.get("body_region_map")
    result = _validate_body_region_map_payload(body_region_map)
    result["details"]["character_id"] = character_id
    return result


def validate_section6_readiness(character_id: str, *, required_region_set: list[str]) -> dict:
    validation_result = validate_body_region_map(character_id)
    details = validation_result.setdefault("details", {})
    details["required_region_set"] = list(required_region_set)

    record = load_character_record(character_id)
    body_region_map = record.get("body_region_map") or {}
    regions = body_region_map.get("regions") or {}

    missing_required_regions = [region_id for region_id in required_region_set if region_id not in regions]
    details["missing_required_regions_for_section7"] = sorted(missing_required_regions)
    if missing_required_regions:
        validation_result.setdefault("errors", []).append(
            f"Required Section 6 regions are missing: {sorted(missing_required_regions)}"
        )

    validation_result["pass"] = len(validation_result.get("errors", [])) == 0
    validation_result["handoff_message"] = SECTION6_READY_MESSAGE if validation_result["pass"] else None
    return validation_result


def finalize_section6_handoff(
    character_id: str,
    *,
    required_region_set: list[str] | None = None,
    source_operation_tag: str = "section6.finalize_handoff",
) -> dict:
    required_region_set = list(required_region_set or SECTION6_REQUIRED_REGION_KEYS)
    validation_result = validate_section6_readiness(character_id, required_region_set=required_region_set)
    if not validation_result["pass"]:
        raise RuntimeError("Section 6 readiness check failed. Resolve errors before continuing to Section 7.")

    record = load_character_record(character_id)
    body_region_map = ensure_body_region_map(character_id)
    regions = body_region_map.get("regions", {})
    for region_id in required_region_set:
        payload = regions.get(region_id)
        if isinstance(payload, dict):
            payload.setdefault("last_edited_at", _utc_now_iso())
            payload.setdefault("source_operation_tag", source_operation_tag)

    body_region_map["regions"] = regions
    body_region_map["updated_at"] = _utc_now_iso()
    body_region_map["section7_handoff"] = {
        "ready": True,
        "message": SECTION6_READY_MESSAGE,
        "finalized_at": _utc_now_iso(),
        "required_region_set": required_region_set,
    }
    record["body_region_map"] = body_region_map
    save_character_record(character_id, record)
    return {
        "pass": True,
        "message": SECTION6_READY_MESSAGE,
        "character_id": character_id,
        "required_region_set": required_region_set,
    }


def get_next_editable_region(character_id: str, preferred_order: list[str] | None = None) -> dict | None:
    body_region_map = ensure_body_region_map(character_id)
    regions = body_region_map.get("regions", {})
    region_order = list(preferred_order or SECTION6_REQUIRED_REGION_KEYS)

    for index, region_id in enumerate(region_order):
        payload = regions.get(region_id)
        if not isinstance(payload, dict):
            continue
        if bool(payload.get("is_locked")) or bool(payload.get("is_frozen")):
            continue
        return {
            "region_id": region_id,
            "order_index": index,
            "region": payload,
        }
    return None


def print_section6_region_completion_table(character_id: str) -> None:
    record = load_character_record(character_id)
    regions = (record.get("body_region_map") or {}).get("regions") or {}

    print("region | mask exists | locked | frozen | last_edited_at | source_operation_tag")
    print("--- | --- | --- | --- | --- | ---")
    for region_id in SECTION6_REQUIRED_REGION_KEYS:
        payload = regions.get(region_id)
        mask_exists = False
        locked_flag = False
        frozen_flag = False
        last_edited_at = None
        source_operation_tag = None
        if isinstance(payload, dict):
            mask_path = payload.get("mask_path")
            if isinstance(mask_path, str) and mask_path.strip():
                mask_exists = Path(mask_path).expanduser().exists()
            locked_flag = bool(payload.get("is_locked"))
            frozen_flag = bool(payload.get("is_frozen"))
            last_edited_at = payload.get("last_edited_at")
            source_operation_tag = payload.get("source_operation_tag")
        print(f"{region_id} | {mask_exists} | {locked_flag} | {frozen_flag} | {last_edited_at} | {source_operation_tag}")



In [ ]:
# Section 6 workflow cell 1 — Input/config
section6_character_id = identity_lock_character_id
section6_source = "notebook.section6"
section6_pose_reference = None  # optional: e.g., "neutral_a_pose"
section6_image_reference = None  # optional: e.g., "/path/to/source_identity.png"
section6_source_image_paths = []  # optional: e.g., ["/path/to/front.png", "/path/to/side.png"]
section6_required_region_set = list(SECTION6_REQUIRED_REGION_KEYS)
section6_reset_body_region_map = False

print("Section 6 config:")
print(json.dumps({
    "character_id": section6_character_id,
    "source": section6_source,
    "pose_reference": section6_pose_reference,
    "image_reference": section6_image_reference,
    "source_image_paths": section6_source_image_paths,
    "required_region_set": section6_required_region_set,
    "reset_body_region_map": section6_reset_body_region_map,
}, indent=2))


In [ ]:
# Section 6 workflow cell 2 — Initialize/reset body_region_map
body_region_map = initialize_body_region_map(
    section6_character_id,
    reset_existing=section6_reset_body_region_map,
    source=section6_source,
    pose_reference=section6_pose_reference,
    image_reference=section6_image_reference,
)
print("Section 6 body_region_map initialized.")
print(json.dumps({"schema_version": body_region_map.get("schema_version"), "region_count": len(body_region_map.get("regions", {}))}, indent=2))


In [ ]:
# Section 6 workflow cell 3 — Register/update mask paths per region
section6_region_masks = {
    # "head": "/path/to/head_mask.png",
    # "chest": "/path/to/chest_mask.png",
}
if section6_region_masks:
    body_region_map = register_body_region_mask_paths(
        section6_character_id,
        section6_region_masks,
        validate_existing=True,
    )
    print(f"Registered/updated masks for {len(section6_region_masks)} region(s).")
else:
    print("No region masks registered in this run. Update section6_region_masks and re-run this cell.")


In [ ]:
# Section 6 workflow cell 4 — Readiness validation gate before Section 7
section6_validation_result = validate_section6_readiness(
    section6_character_id,
    required_region_set=section6_required_region_set,
)
print("Section 6 readiness result:")
print(json.dumps(section6_validation_result, indent=2))

if not section6_validation_result["pass"]:
    raise RuntimeError(
        "Section 6 readiness check failed. Resolve errors before continuing to Section 7."
    )

section6_handoff_result = finalize_section6_handoff(
    section6_character_id,
    required_region_set=section6_required_region_set,
)
print(section6_handoff_result["message"])



In [ ]:
# Section 6 workflow cell 5 — Compact completion summary table
print_section6_region_completion_table(section6_character_id)


---

# SECTION 7 — Regional Refinement Mode (Anatomical Sculpting)

**Goal:**
Build a repeatable regional refinement workflow that improves local fidelity (face/hair/torso/etc.) while preserving Section 5 identity lock invariants and Section 6 region map guarantees.

**Inputs:**
- `character_id` + canonical `character.json`
- Section 6 region map and masks
- Candidate image path
- Optional per-region config overrides

**Outputs:**
- Refined image(s)
- Optional per-region intermediate artifacts
- Validation report (identity, mask coverage, regional quality gates)
- Character record updates with provenance/versioning

**Core functions (notebook first, script mirror):**
- `load_region_map(character_id)`
- `validate_region_map(region_map, masks_dir)`
- `run_regional_refinement(character_id, input_image_path, config=None)`
- `validate_regional_refinement(character_id, refined_image_path, strict=True)`
- `promote_refined_candidate(character_id, refined_image_path, report)`

**Gate policy:**
- Gate A (hard fail): identity lock pass required
- Gate B (hard fail): required regions processed
- Gate C (configurable hard/soft fail): regional quality checks
- Gate D (hard fail): provenance completeness
- Summary status: `PASS`, `PASS_WITH_WARNINGS`, `FAIL`

**Initial region scope:**
- `face_primary`
- `hair_silhouette`
- `upper_torso`

**Implementation sequence:**
1. Finalize Section 6 schema and validators
2. Add append-only `regional_refinement` block in `character.json`
3. Implement map/mask load + validation helpers
4. Implement deterministic regional refinement/compositing
5. Apply Section 5 identity backend fallback (`insightface -> clip -> phash/dhash`)
6. Emit report + promotion metadata
7. Run QA with 3-5 characters under varied pose/lighting


In [ ]:
# SECTION 7 — Regional Refinement Mode (Distinctive Anchor: S7_REGIONAL_REFINEMENT_PIPELINE_V1)
SECTION7_REFINEMENT_SCHEMA_VERSION = "section7.regional_refinement.v1"


def _ensure_section7_refinement_state(record: dict, character_id: str) -> dict:
    section7_state = record.get("section7_refinement")
    if not isinstance(section7_state, dict):
        section7_state = {}

    section7_state.setdefault("schema_version", SECTION7_REFINEMENT_SCHEMA_VERSION)
    section7_state.setdefault("character_id", character_id)
    section7_state.setdefault("active_region", None)
    section7_state.setdefault("completed_regions", [])
    section7_state.setdefault("history", [])
    section7_state.setdefault("created_at", _utc_now_iso())
    section7_state["updated_at"] = _utc_now_iso()
    record["section7_refinement"] = section7_state
    return section7_state


def lock_entire_body_for_refinement(
    character_id: str,
    *,
    source_operation_tag: str = "section7.lock_entire_body",
) -> dict:
    record = load_character_record(character_id)
    body_region_map = ensure_body_region_map(character_id)
    regions = body_region_map.get("regions", {})

    for region_id, payload in regions.items():
        if not isinstance(payload, dict):
            continue
        payload["is_locked"] = True
        payload["last_edited_at"] = _utc_now_iso()
        payload["source_operation_tag"] = source_operation_tag
        regions[region_id] = payload

    body_region_map["regions"] = regions
    body_region_map["updated_at"] = _utc_now_iso()
    record["body_region_map"] = body_region_map

    section7_state = _ensure_section7_refinement_state(record, character_id)
    section7_state["active_region"] = None
    section7_state["history"].append({
        "timestamp": _utc_now_iso(),
        "event": "lock_entire_body",
        "source_operation_tag": source_operation_tag,
    })
    section7_state["updated_at"] = _utc_now_iso()
    record["section7_refinement"] = section7_state

    save_character_record(character_id, record)
    return {"character_id": character_id, "locked_region_count": len(regions)}


def get_next_unrefined_region(character_id: str, preferred_order: list[str] | None = None) -> str | None:
    body_region_map = ensure_body_region_map(character_id)
    regions = body_region_map.get("regions", {})
    order = list(preferred_order or SECTION6_REQUIRED_REGION_KEYS)
    for region_id in order:
        payload = regions.get(region_id)
        if not isinstance(payload, dict):
            continue
        if bool(payload.get("is_frozen")):
            continue
        return region_id
    return None


def unlock_region_for_refinement(
    character_id: str,
    region_id: str,
    *,
    source_operation_tag: str = "section7.unlock_region",
) -> dict:
    record = load_character_record(character_id)
    body_region_map = ensure_body_region_map(character_id)
    regions = body_region_map.get("regions", {})

    if region_id not in regions or not isinstance(regions[region_id], dict):
        raise KeyError(f"Unknown region_id for Section 7 unlock: {region_id}")
    if bool(regions[region_id].get("is_frozen")):
        raise RuntimeError(f"Region '{region_id}' is frozen and cannot be unlocked.")

    for candidate_region_id, payload in regions.items():
        if not isinstance(payload, dict):
            continue
        payload["is_locked"] = candidate_region_id != region_id
        payload["last_edited_at"] = _utc_now_iso()
        payload["source_operation_tag"] = source_operation_tag
        regions[candidate_region_id] = payload

    body_region_map["regions"] = regions
    body_region_map["updated_at"] = _utc_now_iso()
    record["body_region_map"] = body_region_map

    section7_state = _ensure_section7_refinement_state(record, character_id)
    section7_state["active_region"] = region_id
    section7_state["history"].append({
        "timestamp": _utc_now_iso(),
        "event": "unlock_region",
        "region_id": region_id,
        "source_operation_tag": source_operation_tag,
    })
    section7_state["updated_at"] = _utc_now_iso()
    record["section7_refinement"] = section7_state

    save_character_record(character_id, record)
    return {"character_id": character_id, "active_region": region_id}


def record_region_refinement_adjustment(
    character_id: str,
    region_id: str,
    *,
    adjustment_summary: str,
    bounds_check: str,
    source_operation_tag: str = "section7.record_adjustment",
) -> dict:
    if not adjustment_summary.strip():
        raise ValueError("adjustment_summary must be non-empty.")
    if not bounds_check.strip():
        raise ValueError("bounds_check must be non-empty.")

    record = load_character_record(character_id)
    body_region_map = ensure_body_region_map(character_id)
    regions = body_region_map.get("regions", {})
    payload = regions.get(region_id)
    if not isinstance(payload, dict):
        raise KeyError(f"Unknown region_id for Section 7 adjustment: {region_id}")
    if bool(payload.get("is_locked")):
        raise RuntimeError(f"Region '{region_id}' is locked. Unlock it before recording adjustment.")

    adjustments = payload.get("refinement_adjustments")
    if not isinstance(adjustments, list):
        adjustments = []
    adjustment_entry = {
        "timestamp": _utc_now_iso(),
        "summary": adjustment_summary.strip(),
        "bounds_check": bounds_check.strip(),
        "source_operation_tag": source_operation_tag,
    }
    adjustments.append(adjustment_entry)
    payload["refinement_adjustments"] = adjustments
    payload["last_edited_at"] = _utc_now_iso()
    payload["source_operation_tag"] = source_operation_tag
    regions[region_id] = payload

    body_region_map["regions"] = regions
    body_region_map["updated_at"] = _utc_now_iso()
    record["body_region_map"] = body_region_map

    section7_state = _ensure_section7_refinement_state(record, character_id)
    section7_state["active_region"] = region_id
    section7_state["history"].append({
        "timestamp": _utc_now_iso(),
        "event": "record_adjustment",
        "region_id": region_id,
        "bounds_check": bounds_check.strip(),
        "source_operation_tag": source_operation_tag,
    })
    section7_state["updated_at"] = _utc_now_iso()
    record["section7_refinement"] = section7_state

    save_character_record(character_id, record)
    return adjustment_entry


def freeze_refined_region(
    character_id: str,
    region_id: str,
    *,
    source_operation_tag: str = "section7.freeze_region",
) -> dict:
    record = load_character_record(character_id)
    body_region_map = ensure_body_region_map(character_id)
    regions = body_region_map.get("regions", {})
    payload = regions.get(region_id)
    if not isinstance(payload, dict):
        raise KeyError(f"Unknown region_id for Section 7 freeze: {region_id}")

    payload["is_locked"] = True
    payload["is_frozen"] = True
    payload["last_edited_at"] = _utc_now_iso()
    payload["source_operation_tag"] = source_operation_tag
    regions[region_id] = payload

    body_region_map["regions"] = regions
    body_region_map["updated_at"] = _utc_now_iso()
    record["body_region_map"] = body_region_map

    section7_state = _ensure_section7_refinement_state(record, character_id)
    completed_regions = section7_state.get("completed_regions")
    if not isinstance(completed_regions, list):
        completed_regions = []
    if region_id not in completed_regions:
        completed_regions.append(region_id)
    section7_state["completed_regions"] = completed_regions
    section7_state["active_region"] = None
    section7_state["history"].append({
        "timestamp": _utc_now_iso(),
        "event": "freeze_region",
        "region_id": region_id,
        "source_operation_tag": source_operation_tag,
    })
    section7_state["updated_at"] = _utc_now_iso()
    record["section7_refinement"] = section7_state

    save_character_record(character_id, record)
    return {"character_id": character_id, "frozen_region": region_id, "completed_regions": completed_regions}


def get_section7_refinement_status(character_id: str, preferred_order: list[str] | None = None) -> dict:
    record = load_character_record(character_id)
    body_region_map = ensure_body_region_map(character_id)
    regions = body_region_map.get("regions", {})
    section7_state = _ensure_section7_refinement_state(record, character_id)

    order = list(preferred_order or SECTION6_REQUIRED_REGION_KEYS)
    next_region = get_next_unrefined_region(character_id, preferred_order=order)
    frozen_regions = [region_id for region_id in order if bool((regions.get(region_id) or {}).get("is_frozen"))]

    status = {
        "character_id": character_id,
        "active_region": section7_state.get("active_region"),
        "completed_regions": list(section7_state.get("completed_regions") or []),
        "frozen_region_count": len(frozen_regions),
        "remaining_region_count": max(len(order) - len(frozen_regions), 0),
        "next_region": next_region,
        "workflow_complete": next_region is None,
    }
    save_character_record(character_id, record)
    return status

SECTION7_DEFAULT_REGION_SCOPE = ["head", "shoulders", "chest"]
SECTION7_DEFAULT_QUALITY_THRESHOLDS = {
    "identity": 1.0,
    "coverage": 1.0,
    "provenance": 1.0,
}


def load_region_map(character_id: str) -> dict:
    body_region_map = ensure_body_region_map(character_id)
    validation = validate_section6_readiness(character_id)
    required_regions = validation.get("required_region_set") or list(SECTION7_DEFAULT_REGION_SCOPE)
    regions = body_region_map.get("regions", {})
    scoped_regions = {
        region_id: regions[region_id]
        for region_id in required_regions
        if isinstance(regions.get(region_id), dict)
    }
    return {
        "character_id": character_id,
        "schema_version": body_region_map.get("schema_version"),
        "mask_root": body_region_map.get("mask_root", SECTION6_REGION_MASK_SUBDIR),
        "required_regions": required_regions,
        "regions": scoped_regions,
    }


def validate_region_map(region_map: dict, masks_dir: str | Path) -> dict:
    masks_root = Path(masks_dir).expanduser()
    errors = []
    warnings = []
    checked_regions = []
    regions = region_map.get("regions", {}) if isinstance(region_map, dict) else {}

    for region_id in region_map.get("required_regions", []):
        payload = regions.get(region_id)
        if not isinstance(payload, dict):
            errors.append(f"Missing region payload: {region_id}")
            continue

        mask_path = payload.get("mask_path")
        if not isinstance(mask_path, str) or not mask_path.strip():
            errors.append(f"Missing mask_path for region: {region_id}")
            continue

        resolved_mask = (masks_root / mask_path).resolve() if not Path(mask_path).is_absolute() else Path(mask_path).resolve()
        if not resolved_mask.exists():
            errors.append(f"Mask does not exist for region '{region_id}': {resolved_mask}")
            continue

        checked_regions.append(region_id)
        metadata = payload.get("mask_metadata")
        if not isinstance(metadata, dict):
            warnings.append(f"mask_metadata missing for region: {region_id}")

    return {
        "pass": not errors,
        "checked_region_count": len(checked_regions),
        "checked_regions": checked_regions,
        "errors": errors,
        "warnings": warnings,
    }


def run_regional_refinement(character_id: str, input_image_path: str, config: dict | None = None) -> dict:
    config = dict(config or {})
    record = load_character_record(character_id)
    character_dir = _character_dir(character_id)
    region_map = load_region_map(character_id)
    region_validation = validate_region_map(region_map, character_dir)
    if not region_validation["pass"]:
        raise RuntimeError(f"Region map validation failed: {region_validation['errors']}")

    source = Path(input_image_path).expanduser()
    if not source.exists():
        raise FileNotFoundError(f"Regional refinement input image not found: {source}")

    run_id = f"rr-{datetime.utcnow().strftime('%Y%m%dT%H%M%S')}"
    output_name = f"{run_id}_refined{source.suffix.lower() or '.png'}"
    output_rel, output_sha = _copy_to_character_subdir(
        character_dir,
        source,
        "regional_refinement/candidates",
        output_name,
    )

    regional_refinement = record.get("regional_refinement")
    if not isinstance(regional_refinement, dict):
        regional_refinement = {
            "schema_version": "section7.regional_refinement_plan.v1",
            "created_at": _utc_now_iso(),
            "runs": [],
            "promotions": [],
        }

    run_entry = {
        "run_id": run_id,
        "created_at": _utc_now_iso(),
        "input_image_path": str(source.resolve()),
        "output_image": {
            "path": output_rel,
            "sha256": output_sha,
        },
        "config": config,
        "required_regions": list(region_map.get("required_regions") or []),
        "processed_regions": list(region_validation.get("checked_regions") or []),
        "provenance": {
            "operation": "run_regional_refinement",
            "source_operation_tag": config.get("source_operation_tag", "section7.run_regional_refinement"),
        },
    }
    regional_refinement["runs"].append(run_entry)
    regional_refinement["updated_at"] = _utc_now_iso()
    record["regional_refinement"] = regional_refinement
    save_character_record(character_id, record)
    return run_entry


def validate_regional_refinement(character_id: str, refined_image_path: str, strict: bool = True) -> dict:
    record = load_character_record(character_id)
    character_dir = _character_dir(character_id)
    region_map = load_region_map(character_id)
    region_validation = validate_region_map(region_map, character_dir)
    identity_validation = validate_identity_lock(character_id, refined_image_path)

    gate_a_identity = bool(identity_validation.get("pass"))
    required_regions = list(region_map.get("required_regions") or [])
    processed_regions = list(region_validation.get("checked_regions") or [])
    gate_b_regions = set(required_regions).issubset(set(processed_regions))
    provenance_ok = False
    regional_refinement = record.get("regional_refinement")
    if isinstance(regional_refinement, dict):
        runs = regional_refinement.get("runs")
        if isinstance(runs, list) and runs:
            latest = runs[-1]
            provenance = latest.get("provenance") if isinstance(latest, dict) else None
            provenance_ok = isinstance(provenance, dict) and bool(provenance.get("operation")) and bool(provenance.get("source_operation_tag"))

    quality_score = 1.0 if region_validation["pass"] else 0.0
    threshold = float(SECTION7_DEFAULT_QUALITY_THRESHOLDS["coverage"])
    gate_c_quality = quality_score >= threshold
    gate_d_provenance = provenance_ok

    failures = []
    warnings = []
    if not gate_a_identity:
        failures.append("Gate A failed: identity lock validation did not pass.")
    if not gate_b_regions:
        failures.append("Gate B failed: required regions are not fully processed.")
    if not gate_c_quality:
        message = "Gate C failed: regional quality checks are below threshold."
        if strict:
            failures.append(message)
        else:
            warnings.append(message)
    if not gate_d_provenance:
        failures.append("Gate D failed: provenance is incomplete.")

    summary_status = "PASS"
    if failures:
        summary_status = "FAIL"
    elif warnings:
        summary_status = "PASS_WITH_WARNINGS"

    report = {
        "character_id": character_id,
        "timestamp": _utc_now_iso(),
        "strict": strict,
        "summary_status": summary_status,
        "gates": {
            "gate_a_identity": gate_a_identity,
            "gate_b_required_regions": gate_b_regions,
            "gate_c_quality": gate_c_quality,
            "gate_d_provenance": gate_d_provenance,
        },
        "required_regions": required_regions,
        "processed_regions": processed_regions,
        "identity_validation": identity_validation,
        "region_map_validation": region_validation,
        "failures": failures,
        "warnings": warnings,
    }

    regional_refinement = record.get("regional_refinement")
    if not isinstance(regional_refinement, dict):
        regional_refinement = {
            "schema_version": "section7.regional_refinement_plan.v1",
            "created_at": _utc_now_iso(),
            "runs": [],
            "promotions": [],
        }
    reports = regional_refinement.get("validation_reports")
    if not isinstance(reports, list):
        reports = []
    reports.append(report)
    regional_refinement["validation_reports"] = reports
    regional_refinement["updated_at"] = _utc_now_iso()
    record["regional_refinement"] = regional_refinement
    save_character_record(character_id, record)
    return report


def promote_refined_candidate(character_id: str, refined_image_path: str, report: dict) -> dict:
    if not isinstance(report, dict):
        raise ValueError("report must be a validation report dictionary.")
    if report.get("summary_status") not in {"PASS", "PASS_WITH_WARNINGS"}:
        raise RuntimeError("Refined candidate cannot be promoted because validation did not pass.")

    record = load_character_record(character_id)
    character_dir = _character_dir(character_id)
    source = Path(refined_image_path).expanduser()
    promoted_name = f"{datetime.utcnow().strftime('%Y%m%dT%H%M%S')}_regional_refined{source.suffix.lower() or '.png'}"
    promoted_rel, promoted_sha = _copy_to_character_subdir(
        character_dir,
        source,
        "regional_refinement/promoted",
        promoted_name,
    )

    regional_refinement = record.get("regional_refinement")
    if not isinstance(regional_refinement, dict):
        regional_refinement = {
            "schema_version": "section7.regional_refinement_plan.v1",
            "created_at": _utc_now_iso(),
            "runs": [],
            "promotions": [],
        }
    promotions = regional_refinement.get("promotions")
    if not isinstance(promotions, list):
        promotions = []

    promotion_entry = {
        "timestamp": _utc_now_iso(),
        "promoted_image": {
            "path": promoted_rel,
            "sha256": promoted_sha,
        },
        "validation_summary_status": report.get("summary_status"),
        "report_timestamp": report.get("timestamp"),
    }
    promotions.append(promotion_entry)
    regional_refinement["promotions"] = promotions
    regional_refinement["latest_promoted_image"] = promoted_rel
    regional_refinement["updated_at"] = _utc_now_iso()
    record["regional_refinement"] = regional_refinement
    save_character_record(character_id, record)
    return promotion_entry


In [ ]:
# Section 7 workflow cell 1 — Input/config
section7_character_id = section6_character_id
section7_preferred_order = list(SECTION6_REQUIRED_REGION_KEYS)
section7_adjustment_summary = ""  # e.g., "Adjusted clavicle width +2% for shoulder alignment"
section7_bounds_check = ""  # e.g., "Within anatomical reference bounds"

print("Section 7 config:")
print(json.dumps({
    "character_id": section7_character_id,
    "preferred_order": section7_preferred_order,
    "adjustment_summary": section7_adjustment_summary,
    "bounds_check": section7_bounds_check,
}, indent=2))


In [ ]:
# Section 7 workflow cell 2 — Lock entire body
section7_lock_result = lock_entire_body_for_refinement(section7_character_id)
print(json.dumps(section7_lock_result, indent=2))


In [ ]:
# Section 7 workflow cell 3 — Unlock next region
section7_next_region = get_next_unrefined_region(
    section7_character_id,
    preferred_order=section7_preferred_order,
)
if section7_next_region is None:
    print("All Section 7 regions are frozen. No region available to unlock.")
else:
    section7_unlock_result = unlock_region_for_refinement(
        section7_character_id,
        section7_next_region,
    )
    print(json.dumps(section7_unlock_result, indent=2))


In [ ]:
# Section 7 workflow cell 4 — Record bounded adjustment and freeze region
if section7_next_region and section7_adjustment_summary and section7_bounds_check:
    section7_adjustment = record_region_refinement_adjustment(
        section7_character_id,
        section7_next_region,
        adjustment_summary=section7_adjustment_summary,
        bounds_check=section7_bounds_check,
    )
    print("Recorded adjustment:")
    print(json.dumps(section7_adjustment, indent=2))

    section7_freeze_result = freeze_refined_region(
        section7_character_id,
        section7_next_region,
    )
    print("Frozen region:")
    print(json.dumps(section7_freeze_result, indent=2))
else:
    print("Set section7_adjustment_summary and section7_bounds_check to record an adjustment and freeze the unlocked region.")


In [ ]:
# Section 7 workflow cell 5 — Move to next region (status preview)
section7_status = get_section7_refinement_status(
    section7_character_id,
    preferred_order=section7_preferred_order,
)
print("Section 7 status:")
print(json.dumps(section7_status, indent=2))


In [ ]:
# SECTION 9 — Pose & Deformation Generation (Distinctive Anchor: S9_POSE_DEFORMATION_SCHEMA_V1)
SECTION9_SCHEMA_VERSION = "section9.pose_deformation.v1"
SECTION9_REQUIRED_STAGE = "section8"
SECTION9_ALLOWED_POSE_LABELS = {
    "standing",
    "sitting",
    "walking",
    "reaching",
    "twisting",
    "bending",
    "flexing",
}

SECTION9_GATE_PASS = "PASS"
SECTION9_GATE_PASS_WITH_WARNINGS = "PASS_WITH_WARNINGS"
SECTION9_GATE_FAIL = "FAIL"


SECTION10_SCHEMA_VERSION = "section10.lighting_camera_reference.v1"
SECTION10_REQUIRED_STAGE = "section8"
SECTION10_ALLOWED_LIGHTING_PRESETS = {
    "neutral_studio",
    "soft_key",
    "rim_light",
    "dramatic_split",
    "outdoor_overcast",
}
SECTION10_ALLOWED_VIEW_LABELS = {
    "front",
    "three_quarter_left",
    "three_quarter_right",
    "profile_left",
    "profile_right",
}
SECTION10_ALLOWED_CAMERA_PROFILE_TYPES = {
    "portrait_85mm",
    "standard_50mm",
    "wide_35mm",
    "orthographic",
}


def _ensure_section9_pose_state(record: dict, character_id: str) -> dict:
    section9_state = record.get("section9_pose_deformation")
    if not isinstance(section9_state, dict):
        section9_state = {}

    section9_state.setdefault("schema_version", SECTION9_SCHEMA_VERSION)
    section9_state.setdefault("character_id", character_id)
    section9_state.setdefault("runs", [])
    section9_state.setdefault("latest_run_id", None)
    section9_state.setdefault("created_at", _utc_now_iso())
    section9_state["updated_at"] = _utc_now_iso()
    record["section9_pose_deformation"] = section9_state
    return section9_state


def _assert_section9_lifecycle_gate(record: dict) -> None:
    section8_state = record.get("section8_canonical_finalization")
    section8_anatomy_state = section8_state.get("anatomy_state") if isinstance(section8_state, dict) else None
    lifecycle_state = (record.get("lifecycle") or {}).get("anatomy_state")
    anatomy_state = section8_anatomy_state or lifecycle_state

    if anatomy_state != "canonical_frozen":
        raise RuntimeError(
            "Section 9 requires Section 8 canonical freeze. "
            "Run Section 8 finalization until anatomy_state == 'canonical_frozen'."
        )


def _resolve_section9_canonical_source(record: dict, character_dir: Path, source_canonical_image: str | None = None) -> tuple[str | None, Path | None]:
    if source_canonical_image:
        source_path = Path(source_canonical_image).expanduser()
        if source_path.is_absolute():
            return str(source_path), source_path
        return source_canonical_image, (character_dir / source_path)

    section8_state = record.get("section8_canonical_finalization")
    if isinstance(section8_state, dict):
        latest_rel = section8_state.get("latest_canonical_image")
        if isinstance(latest_rel, str) and latest_rel.strip():
            return latest_rel, (character_dir / latest_rel)
    return None, None


def _collect_section9_identity_artifact_issues(identity_lock: dict, character_dir: Path) -> list[str]:
    issues = []

    references = identity_lock.get("reference_images")
    if not isinstance(references, list) or not references:
        issues.append("identity_lock.reference_images is missing or empty.")
    else:
        for idx, reference in enumerate(references, start=1):
            rel_path = reference.get("file") if isinstance(reference, dict) else None
            if not isinstance(rel_path, str) or not rel_path.strip():
                issues.append(f"identity_lock.reference_images[{idx - 1}].file is missing.")
                continue
            if not (character_dir / rel_path).exists():
                issues.append(f"identity lock reference image does not exist: {rel_path}")

    features = ((identity_lock.get("invariants") or {}).get("features") or [])
    if isinstance(features, list):
        for idx, feature in enumerate(features, start=1):
            if not isinstance(feature, dict):
                issues.append(f"identity_lock.invariants.features[{idx - 1}] must be a dict.")
                continue
            mask_file = feature.get("mask_file")
            if isinstance(mask_file, str) and mask_file.strip() and not (character_dir / mask_file).exists():
                issues.append(f"identity invariant mask does not exist: {mask_file}")

    return issues


def _resolve_section9_pose_assets(character_dir: Path, pose_spec: dict) -> list[dict]:
    resolved_assets = []
    for key, value in (pose_spec or {}).items():
        if key in {"pose_id", "pose_label", "prompt", "negative_prompt", "seed"}:
            continue

        candidate_paths = []
        if isinstance(value, str):
            candidate_paths = [value]
        elif isinstance(value, list):
            candidate_paths = [item for item in value if isinstance(item, str)]
        elif isinstance(value, dict):
            candidate_paths = [item for item in value.values() if isinstance(item, str)]

        for raw_path in candidate_paths:
            source_path = Path(raw_path).expanduser()
            if not source_path.is_absolute():
                source_path = character_dir / source_path
            resolved_assets.append({
                "key": key,
                "source": raw_path,
                "resolved_path": str(source_path),
                "exists": source_path.exists(),
            })
    return resolved_assets


def _invoke_section9_generation_adapter(
    character_id: str,
    canonical_source_path: Path,
    pose_spec: dict,
    resolved_assets: list[dict],
    output_path: Path,
) -> dict:
    del character_id, pose_spec, resolved_assets
    output_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(canonical_source_path, output_path)
    return {
        "adapter": "placeholder",
        "mode": "copy_canonical_source",
        "output_path": str(output_path),
        "note": "Replace with real generation adapter when Section 9 contracts are finalized.",
    }


def evaluate_section9_run(report: dict) -> dict:
    if not isinstance(report, dict):
        raise TypeError("report must be a dictionary")

    failures = []
    warnings = []

    validation = report.get("validation") if isinstance(report.get("validation"), dict) else {}
    identity_pass = validation.get("identity_pass")
    if identity_pass is None and isinstance(validation.get("report"), dict):
        identity_pass = validation["report"].get("pass")
    if identity_pass is False:
        failures.append("Identity lock validation failed.")

    provenance = report.get("provenance") if isinstance(report.get("provenance"), dict) else {}
    source_tag = provenance.get("source_operation_tag") or report.get("source_operation_tag")

    input_payload = report.get("input") if isinstance(report.get("input"), dict) else {}
    output_payload = report.get("output") if isinstance(report.get("output"), dict) else {}
    source_info = input_payload.get("canonical_image_path") or report.get("source_canonical_image")
    input_hash = input_payload.get("canonical_image_sha256")

    output_hash_present = isinstance(output_payload.get("sha256"), str) and bool(output_payload.get("sha256"))
    outputs = report.get("outputs") if isinstance(report.get("outputs"), list) else []
    outputs_have_hashes = bool(outputs) and all(
        isinstance(item, dict) and isinstance(item.get("hash"), str) and bool(item.get("hash"))
        for item in outputs
    )

    missing_provenance_fields = []
    if not isinstance(source_tag, str) or not source_tag.strip():
        missing_provenance_fields.append("source_operation_tag")
    if not isinstance(source_info, str) or not source_info.strip():
        missing_provenance_fields.append("input/source")
    if not isinstance(input_hash, str) or not input_hash.strip():
        missing_provenance_fields.append("input/hash")
    if not output_hash_present and not outputs_have_hashes:
        missing_provenance_fields.append("output/hash")

    if missing_provenance_fields:
        failures.append(f"Missing provenance fields: {missing_provenance_fields}")

    unresolved_required_artifacts = []
    resolved_assets = input_payload.get("resolved_assets") if isinstance(input_payload.get("resolved_assets"), list) else []
    unresolved_required_artifacts.extend(
        item.get("resolved_path")
        for item in resolved_assets
        if isinstance(item, dict) and item.get("exists") is False
    )

    request_validation = validation.get("request") if isinstance(validation.get("request"), dict) else {}
    request_failures = request_validation.get("failures") if isinstance(request_validation.get("failures"), list) else []
    unresolved_required_artifacts.extend(
        failure for failure in request_failures if isinstance(failure, str) and "artifact" in failure.lower()
    )

    if unresolved_required_artifacts:
        failures.append(f"Unresolved required artifacts: {unresolved_required_artifacts}")

    anatomy_heuristics = report.get("anatomy_heuristics")
    if anatomy_heuristics is None:
        anatomy_heuristics = validation.get("anatomy_heuristics")
    if isinstance(anatomy_heuristics, dict):
        heuristics_failed = anatomy_heuristics.get("pass") is False
        heuristics_failures = anatomy_heuristics.get("failures") if isinstance(anatomy_heuristics.get("failures"), list) else []
        if heuristics_failed or heuristics_failures:
            warnings.append(
                f"Optional anatomy heuristics reported issues: {heuristics_failures or ['anatomy_heuristics.pass == False']}"
            )

    gate_status = SECTION9_GATE_FAIL if failures else (
        SECTION9_GATE_PASS_WITH_WARNINGS if warnings else SECTION9_GATE_PASS
    )
    return {
        "gate_status": gate_status,
        "gate_failures": failures,
        "gate_warnings": warnings,
        "decision_timestamp": _utc_now_iso(),
    }


def validate_section9_pose_request(
    character_id: str,
    *,
    pose_id: str,
    pose_label: str,
    source_canonical_image: str | None = None,
) -> dict:
    record = load_character_record(character_id)
    character_dir = _character_dir(character_id)

    failures = []
    warnings = []

    normalized_pose_id = str(pose_id or "").strip()
    normalized_pose_label = str(pose_label or "").strip().lower()
    if not normalized_pose_id or not normalized_pose_label:
        failures.append("Pose definition must include non-empty pose_id and pose_label.")
    elif normalized_pose_label not in SECTION9_ALLOWED_POSE_LABELS:
        failures.append(
            f"pose_label '{pose_label}' is not supported. Allowed labels: {sorted(SECTION9_ALLOWED_POSE_LABELS)}"
        )

    section8_state = record.get("section8_canonical_finalization")
    finalizations = section8_state.get("finalizations") if isinstance(section8_state, dict) else None
    is_finalized = isinstance(finalizations, list) and bool(finalizations)
    if not is_finalized:
        failures.append("Section 8 finalized status must be true before Section 9 runs.")

    canonical_rel, canonical_path = _resolve_section9_canonical_source(record, character_dir, source_canonical_image)
    if canonical_path is None or not canonical_path.exists():
        failures.append("Canonical source image does not exist for Section 9 pose request.")

    identity_lock = record.get("identity_lock")
    if not isinstance(identity_lock, dict):
        failures.append("identity_lock is missing; required identity lock artifacts are unavailable.")
        identity_artifact_issues = []
    else:
        identity_artifact_issues = _collect_section9_identity_artifact_issues(identity_lock, character_dir)
        failures.extend(identity_artifact_issues)

    return {
        "character_id": character_id,
        "timestamp": _utc_now_iso(),
        "pose_definition": {
            "pose_id": normalized_pose_id,
            "pose_label": normalized_pose_label,
        },
        "source_canonical_image": canonical_rel,
        "canonical_source_exists": bool(canonical_path and canonical_path.exists()),
        "section8_finalized": is_finalized,
        "identity_artifacts_present": not identity_artifact_issues,
        "warnings": warnings,
        "failures": failures,
        "pass": not failures,
    }


def create_section9_pose_deformation_run(
    character_id: str,
    *,
    pose_id: str,
    pose_label: str,
    source_canonical_image: str | None = None,
    deformation_policy: dict | None = None,
    identity_policy: dict | None = None,
    outputs: list[dict] | None = None,
    source_operation_tag: str = "section9.pose_deformation",
) -> dict:
    record = load_character_record(character_id)
    _assert_section9_lifecycle_gate(record)
    section9_state = _ensure_section9_pose_state(record, character_id)

    request_validation = validate_section9_pose_request(
        character_id,
        pose_id=pose_id,
        pose_label=pose_label,
        source_canonical_image=source_canonical_image,
    )
    if not request_validation["pass"]:
        raise RuntimeError(f"Invalid Section 9 pose request: {request_validation['failures']}")

    character_dir = _character_dir(character_id)
    canonical_rel, canonical_path = _resolve_section9_canonical_source(record, character_dir, source_canonical_image)

    run_id = f"s9_{datetime.utcnow().strftime('%Y%m%dT%H%M%S')}"
    outputs_entries = []
    drift_reasons = []
    identity_checks = []

    for output in list(outputs or []):
        if not isinstance(output, dict):
            continue
        image_path = output.get("image_path")
        if not isinstance(image_path, str) or not image_path.strip():
            continue
        output_path = Path(image_path).expanduser()
        output_exists = output_path.exists()
        output_entry = {
            "image_path": str(output_path),
            "hash": _sha256_file(output_path) if output_exists else None,
            "metadata": dict(output.get("metadata") or {}),
        }
        outputs_entries.append(output_entry)

        if output_exists:
            identity_validation = validate_identity_lock(character_id, str(output_path))
            identity_checks.append({
                "image_path": str(output_path),
                "pass": bool(identity_validation.get("pass")),
                "drift_reasons": list(identity_validation.get("reasons") or []),
            })
            if not identity_validation.get("pass"):
                drift_reasons.extend(identity_validation.get("reasons") or ["Identity validation failed."])
        else:
            reason = f"Output image does not exist: {output_path}"
            drift_reasons.append(reason)
            identity_checks.append({
                "image_path": str(output_path),
                "pass": False,
                "drift_reasons": [reason],
            })

    run_entry = {
        "run_id": run_id,
        "timestamp": _utc_now_iso(),
        "pose_id": request_validation["pose_definition"]["pose_id"],
        "pose_label": request_validation["pose_definition"]["pose_label"],
        "source_canonical_image": canonical_rel,
        "deformation_policy": dict(deformation_policy or {}),
        "identity_policy": dict(identity_policy or {"strict_lock_checks": True}),
        "outputs": outputs_entries,
        "validation": {
            "request": request_validation,
            "identity_pass": all(check.get("pass") for check in identity_checks) if identity_checks else True,
            "drift_reasons": drift_reasons,
            "identity_checks": identity_checks,
        },
        "required_stage": SECTION9_REQUIRED_STAGE,
        "source_operation_tag": source_operation_tag,
    }
    run_entry.update(evaluate_section9_run(run_entry))

    runs = section9_state.get("runs")
    if not isinstance(runs, list):
        runs = []
    runs.append(run_entry)

    section9_state["runs"] = runs
    section9_state["latest_run_id"] = run_id
    section9_state["updated_at"] = _utc_now_iso()
    record["section9_pose_deformation"] = section9_state
    save_character_record(character_id, record)
    return run_entry


def run_pose_deformation_generation(
    character_id: str,
    canonical_image_path: str,
    pose_spec: dict,
    *,
    notes: str | None = None,
    source_operation_tag: str = "section9.pose_deformation",
) -> dict:
    if not isinstance(pose_spec, dict):
        raise TypeError("pose_spec must be a dictionary")

    record = load_character_record(character_id)
    _assert_section9_lifecycle_gate(record)
    section9_state = _ensure_section9_pose_state(record, character_id)

    section8_state = record.get("section8_canonical_finalization")
    finalizations = section8_state.get("finalizations") if isinstance(section8_state, dict) else None
    if not isinstance(finalizations, list) or not finalizations:
        raise RuntimeError("Section 8 finalization history is missing; cannot run Section 9 generation.")

    character_dir = _character_dir(character_id)
    canonical_source = Path(canonical_image_path).expanduser()
    if not canonical_source.is_absolute():
        canonical_source = character_dir / canonical_source
    if not canonical_source.exists():
        raise FileNotFoundError(f"Canonical image does not exist: {canonical_source}")

    resolved_assets = _resolve_section9_pose_assets(character_dir, pose_spec)
    missing_assets = [item for item in resolved_assets if not item.get("exists")]
    if missing_assets:
        missing_paths = [item["resolved_path"] for item in missing_assets]
        raise FileNotFoundError(f"Section 9 pose assets not found: {missing_paths}")

    pose_id = str(pose_spec.get("pose_id") or "pose-unspecified").strip() or "pose-unspecified"
    pose_label = str(pose_spec.get("pose_label") or "custom").strip().lower() or "custom"
    run_id = f"s9_{datetime.utcnow().strftime('%Y%m%dT%H%M%S')}"

    output_ext = canonical_source.suffix.lower() or ".png"
    output_name = f"{run_id}_{pose_id.lower().replace(' ', '_')}{output_ext}"
    output_rel = Path("canonical") / "pose_deformations" / output_name
    output_abs = character_dir / output_rel

    adapter_report = _invoke_section9_generation_adapter(
        character_id,
        canonical_source,
        pose_spec,
        resolved_assets,
        output_abs,
    )
    output_sha = _sha256_file(output_abs)
    identity_validation = validate_identity_lock(character_id, str(output_abs))

    run_entry = {
        "run_id": run_id,
        "timestamp": _utc_now_iso(),
        "pose_spec": dict(pose_spec),
        "pose_id": pose_id,
        "pose_label": pose_label,
        "notes": notes,
        "input": {
            "canonical_image_path": str(canonical_source),
            "canonical_image_sha256": _sha256_file(canonical_source),
            "resolved_assets": resolved_assets,
        },
        "output": {
            "image_path": str(output_rel),
            "sha256": output_sha,
        },
        "validation": {
            "identity_pass": bool(identity_validation.get("pass")),
            "backend_used": identity_validation.get("backend_used"),
            "summary": {
                "reasons": list(identity_validation.get("reasons") or []),
                "metrics": dict(identity_validation.get("metrics") or {}),
            },
            "report": identity_validation,
        },
        "provenance": {
            "operation": "run_pose_deformation_generation",
            "source_operation_tag": source_operation_tag,
            "section_required_stage": SECTION9_REQUIRED_STAGE,
            "adapter": adapter_report,
        },
    }
    run_entry.update(evaluate_section9_run(run_entry))

    runs = section9_state.get("runs")
    if not isinstance(runs, list):
        runs = []
    runs.append(run_entry)
    section9_state["runs"] = runs
    section9_state["latest_run_id"] = run_id
    section9_state["updated_at"] = _utc_now_iso()
    record["section9_pose_deformation"] = section9_state

    save_character_record(character_id, record)
    return run_entry


def get_section9_pose_deformation_status(character_id: str) -> dict:
    record = load_character_record(character_id)
    section9_state = _ensure_section9_pose_state(record, character_id)

    runs = section9_state.get("runs")
    if not isinstance(runs, list):
        runs = []

    anatomy_state = (
        (record.get("section8_canonical_finalization") or {}).get("anatomy_state")
        or (record.get("lifecycle") or {}).get("anatomy_state")
        or "pre_finalization"
    )

    status = {
        "character_id": character_id,
        "schema_version": section9_state.get("schema_version"),
        "required_stage": SECTION9_REQUIRED_STAGE,
        "anatomy_state": anatomy_state,
        "gate_pass": anatomy_state == "canonical_frozen",
        "run_count": len(runs),
        "latest_run_id": section9_state.get("latest_run_id"),
    }
    save_character_record(character_id, record)
    return status


def get_section9_pose_status(character_id: str) -> dict:
    record = load_character_record(character_id)
    section9_state = _ensure_section9_pose_state(record, character_id)

    runs = section9_state.get("runs")
    if not isinstance(runs, list):
        runs = []

    gate_counts = {"PASS": 0, "PASS_WITH_WARNINGS": 0, "FAIL": 0}
    for run in runs:
        if not isinstance(run, dict):
            continue
        gate_status = str(((run.get("gate") or {}).get("status") or "")).upper()
        if gate_status in gate_counts:
            gate_counts[gate_status] += 1

    latest_run = runs[-1] if runs and isinstance(runs[-1], dict) else {}
    latest_gate_status = (latest_run.get("gate") or {}).get("status")
    latest_pose_label = latest_run.get("pose_label")

    status = {
        "character_id": character_id,
        "total_runs": len(runs),
        "pass_count": gate_counts["PASS"],
        "warn_count": gate_counts["PASS_WITH_WARNINGS"],
        "fail_count": gate_counts["FAIL"],
        "latest_run_id": latest_run.get("run_id"),
        "latest_pose_label": latest_pose_label,
        "latest_gate_status": latest_gate_status,
    }
    save_character_record(character_id, record)
    return status


def _ensure_section10_reference_state(record: dict, character_id: str) -> dict:
    section10_state = record.get("section10_lighting_camera_reference")
    if not isinstance(section10_state, dict):
        section10_state = {}

    section10_state.setdefault("schema_version", SECTION10_SCHEMA_VERSION)
    section10_state.setdefault("character_id", character_id)
    section10_state.setdefault("runs", [])
    section10_state.setdefault("latest_run_id", None)
    section10_state.setdefault("created_at", _utc_now_iso())
    section10_state["updated_at"] = _utc_now_iso()
    record["section10_lighting_camera_reference"] = section10_state
    return section10_state


def _assert_section10_lifecycle_gate(record: dict) -> None:
    section8_state = record.get("section8_canonical_finalization")
    section8_anatomy_state = section8_state.get("anatomy_state") if isinstance(section8_state, dict) else None
    lifecycle_state = (record.get("lifecycle") or {}).get("anatomy_state")
    anatomy_state = section8_anatomy_state or lifecycle_state

    if anatomy_state != "canonical_frozen":
        raise RuntimeError(
            "Section 10 requires Section 8 canonical freeze. "
            "Run Section 8 finalization until anatomy_state == 'canonical_frozen'."
        )


def create_section10_lighting_camera_reference_run(
    character_id: str,
    source_image_path: str,
    *,
    lighting_preset: str,
    view_label: str,
    camera_profile_type: str,
    notes: str | None = None,
    source_operation_tag: str = "section10.lighting_camera_reference",
) -> dict:
    record = load_character_record(character_id)
    _assert_section10_lifecycle_gate(record)
    section10_state = _ensure_section10_reference_state(record, character_id)

    normalized_lighting = str(lighting_preset or "").strip().lower()
    normalized_view = str(view_label or "").strip().lower()
    normalized_camera = str(camera_profile_type or "").strip().lower()

    if normalized_lighting not in SECTION10_ALLOWED_LIGHTING_PRESETS:
        raise ValueError(
            f"lighting_preset '{lighting_preset}' is not supported. "
            f"Allowed presets: {sorted(SECTION10_ALLOWED_LIGHTING_PRESETS)}"
        )
    if normalized_view not in SECTION10_ALLOWED_VIEW_LABELS:
        raise ValueError(
            f"view_label '{view_label}' is not supported. "
            f"Allowed views: {sorted(SECTION10_ALLOWED_VIEW_LABELS)}"
        )
    if normalized_camera not in SECTION10_ALLOWED_CAMERA_PROFILE_TYPES:
        raise ValueError(
            f"camera_profile_type '{camera_profile_type}' is not supported. "
            f"Allowed profiles: {sorted(SECTION10_ALLOWED_CAMERA_PROFILE_TYPES)}"
        )

    if SECTION10_REQUIRED_STAGE == "section9":
        section9_runs = ((record.get("section9_pose_deformation") or {}).get("runs") or [])
        if not isinstance(section9_runs, list) or not section9_runs:
            raise RuntimeError("Section 10 requires at least one Section 9 run when SECTION10_REQUIRED_STAGE='section9'.")

    character_dir = _character_dir(character_id)
    source_path = Path(source_image_path).expanduser()
    if not source_path.is_absolute():
        source_path = character_dir / source_path
    if not source_path.exists():
        raise FileNotFoundError(f"Section 10 source image does not exist: {source_path}")

    run_id = f"s10_{datetime.utcnow().strftime('%Y%m%dT%H%M%S')}"
    output_name = f"{run_id}_{normalized_view}_{normalized_lighting}{source_path.suffix.lower() or '.png'}"
    output_rel, output_sha = _copy_to_character_subdir(
        character_dir,
        source_path,
        "canonical/lighting_camera_reference",
        output_name,
    )

    run_entry = {
        "run_id": run_id,
        "timestamp": _utc_now_iso(),
        "lighting_preset": normalized_lighting,
        "view_label": normalized_view,
        "camera_profile_type": normalized_camera,
        "required_stage": SECTION10_REQUIRED_STAGE,
        "source_image": {
            "path": str(source_path),
            "sha256": _sha256_file(source_path),
        },
        "output": {
            "image_path": output_rel,
            "sha256": output_sha,
        },
        "notes": (notes or "").strip() or None,
        "provenance": {
            "operation": "create_section10_lighting_camera_reference_run",
            "source_operation_tag": source_operation_tag,
        },
    }

    runs = section10_state.get("runs")
    if not isinstance(runs, list):
        runs = []
    runs.append(run_entry)
    section10_state["runs"] = runs
    section10_state["latest_run_id"] = run_id
    section10_state["updated_at"] = _utc_now_iso()
    record["section10_lighting_camera_reference"] = section10_state

    save_character_record(character_id, record)
    return run_entry


def get_section10_lighting_camera_reference_status(character_id: str) -> dict:
    record = load_character_record(character_id)
    section10_state = _ensure_section10_reference_state(record, character_id)

    runs = section10_state.get("runs")
    if not isinstance(runs, list):
        runs = []

    anatomy_state = (
        (record.get("section8_canonical_finalization") or {}).get("anatomy_state")
        or (record.get("lifecycle") or {}).get("anatomy_state")
        or "pre_finalization"
    )

    latest_run = runs[-1] if runs and isinstance(runs[-1], dict) else {}
    status = {
        "character_id": character_id,
        "schema_version": section10_state.get("schema_version"),
        "required_stage": SECTION10_REQUIRED_STAGE,
        "anatomy_state": anatomy_state,
        "gate_pass": anatomy_state == "canonical_frozen",
        "run_count": len(runs),
        "latest_run_id": section10_state.get("latest_run_id"),
        "latest_lighting_preset": latest_run.get("lighting_preset"),
        "latest_view_label": latest_run.get("view_label"),
        "latest_camera_profile_type": latest_run.get("camera_profile_type"),
    }
    save_character_record(character_id, record)
    return status


# Section 9 workflow cell 1 — Input/config (minimal invocation example)
section9_character_id = section8_character_id
section9_canonical_image_path = ""  # required path to canonical frozen image
section9_pose_spec = {
    "pose_id": "",  # e.g., "pose-001"
    "pose_label": "",  # e.g., standing/sitting/walking/reaching/twisting/bending/flexing
}
section9_notes = ""

print("Section 9 config:")
print(json.dumps({
    "character_id": section9_character_id,
    "canonical_image_path": section9_canonical_image_path,
    "pose_spec": section9_pose_spec,
    "notes": section9_notes,
}, indent=2))

# Section 9 workflow cell 2 — Run pose deformation generation (guarded invocation)
if section9_canonical_image_path and isinstance(section9_pose_spec, dict) and section9_pose_spec.get("pose_label"):
    section9_run_result = run_pose_deformation_generation(
        section9_character_id,
        section9_canonical_image_path,
        section9_pose_spec,
        notes=section9_notes or None,
    )
    print("Section 9 run result:")
    print(json.dumps(section9_run_result, indent=2))
else:
    print("Set section9_canonical_image_path and section9_pose_spec['pose_label'] to run Section 9.")

# Section 9 workflow cell 3 — Status preview
section9_status = get_section9_pose_status(section9_character_id)
print("Section 9 status:")
print(json.dumps(section9_status, indent=2))


# Section 10 workflow cell 1 — Input/config (minimal invocation example)
section10_character_id = section9_character_id
section10_source_image_path = ""  # required path to a canonical/pose image to register as reference
section10_lighting_preset = "neutral_studio"
section10_view_label = "front"
section10_camera_profile_type = "portrait_85mm"
section10_notes = ""

print("Section 10 config:")
print(json.dumps({
    "character_id": section10_character_id,
    "source_image_path": section10_source_image_path,
    "lighting_preset": section10_lighting_preset,
    "view_label": section10_view_label,
    "camera_profile_type": section10_camera_profile_type,
    "notes": section10_notes,
}, indent=2))

# Section 10 workflow cell 2 — Register lighting/camera reference (guarded invocation)
if section10_source_image_path:
    section10_run_result = create_section10_lighting_camera_reference_run(
        section10_character_id,
        section10_source_image_path,
        lighting_preset=section10_lighting_preset,
        view_label=section10_view_label,
        camera_profile_type=section10_camera_profile_type,
        notes=section10_notes or None,
    )
    print("Section 10 run result:")
    print(json.dumps(section10_run_result, indent=2))
else:
    print("Set section10_source_image_path to run Section 10.")

# Section 10 workflow cell 3 — Status preview
section10_status = get_section10_lighting_camera_reference_status(section10_character_id)
print("Section 10 status:")
print(json.dumps(section10_status, indent=2))


---

# SECTION 8 — Canonical Body Finalization

**Purpose:**  
Declare the anatomy complete.

Includes:
- Full-body lock
- Cross-pose consistency checks
- Identity snapshot
- Version tagging

After this point, anatomy should not be altered.

---

# SECTION 9 — Pose & Deformation Generation

**Purpose:**  
Generate realistic pose-driven deformation.

Examples:
- Standing
- Sitting
- Walking
- Reaching
- Twisting
- Bending
- Flexing

Rules:
- Deformation allowed
- Identity drift forbidden

---

# SECTION 10 — Lighting, Camera & Reference Views

**Purpose:**  
Create artist-friendly reference images.

Includes:
- Neutral studio lighting
- Orthographic-like views
- Camera angle sweeps
- Optional dramatic lighting


---

# SECTION 11 — Reference Set Export

**Purpose:**  
Export a complete, organized reference set.

Includes:
- Consistent filenames
- Pose labeling
- Lighting variants
- Metadata files

This output is ready for drawing, modeling, or study.


---

# SECTION 12 — Character Save & Reload System

**Purpose:**  
Persist characters across sessions.

Capabilities:
- Save character state
- Reload identity locks
- Resume rendering
- Continue refinement (if unlocked)

Characters become reusable assets.

---

# SECTION 13 — Scene Rendering (Single Character)

**Purpose:**  
Place a character into an environment while preserving identity.

Examples:
- Sitting on a bench
- Walking through a park
- Standing in conversation
- Environmental interaction

---

# SECTION 14 — Multi-Character Scene Composition

**Purpose:**  
Create scenes involving multiple characters.

Workflow:
1. Load characters independently
2. Generate poses separately
3. Match camera and lighting
4. Composite using depth awareness

Example:
- Character B sitting
- Character F approaching and waving

---

# SECTION 15 — Quality Control & Drift Detection

**Purpose:**  
Ensure long-term identity stability.

Includes:
- Embedding similarity checks
- Visual diffs
- Automated rejection rules
- Manual inspection tools


---

# SECTION 16 — Archive, Export & Versioning

**Purpose:**  
Long-term character management.

Includes:
- Version history
- Anatomy revisions
- Export formats
- Backup strategy

---

# SECTION 17 — Notes, Experiments & Future Extensions

**Purpose:**  
A sandbox for:
- Model upgrades
- New controls
- Experimental ideas
- Deferred features

This section keeps the rest of the notebook clean.